<a href="https://colab.research.google.com/github/Takumi173/Test/blob/main/Dataset_JSON_Reviewer_JSON_Refactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 準備

## ファイルの取得とAPI Key設定

In [1]:
# 外部リソースの準備 (一度だけ実行)
!git clone https://github.com/cdisc-org/sdtm-adam-pilot-project.git
!pip install -q pypdf google-generativeai
!wget -q https://github.com/Takumi173/Test/releases/download/testdata/Lzzt_protocol_redacted.pdf

Cloning into 'sdtm-adam-pilot-project'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 224 (delta 64), reused 220 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 24.51 MiB | 2.71 MiB/s, done.
Resolving deltas: 100% (64/64), done.
Updating files: 100% (87/87), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 11.1 MB/s eta 0:00:00


In [2]:
# ライブラリのインポート
import os
import shutil
import json
import copy
import time
import sys
from typing import List, Dict, Any, Tuple, Optional

# データ処理・LLM関連ライブラリ
import pypdf
import google.generativeai as genai
from google.colab import userdata

# APIキーの設定 (自身の環境に合わせてキー名を設定してください)
# api_key = userdata.get('GOOGLE_API_KEY')
# api_key = userdata.get('GOOGLE_API_KEY2')
api_key = userdata.get('GOOGLE_API_KEY_893') # 例
genai.configure(api_key=api_key)

print("インポートと外部リソース準備が完了しました。")

インポートと外部リソース準備が完了しました。


In [3]:
# 使用するjsonデータとdefine.xmlを新規フォルダにコピーする

source_dir = "sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm"
json_dir   = "json_files"
define_dir = "define_xml"

if not os.path.exists(json_dir):
    os.makedirs(json_dir)

if not os.path.exists(define_dir):
    os.makedirs(define_dir)

for root, _, files in os.walk(source_dir):
  for file in files:
    if file.endswith(".json"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(json_dir, file)
      shutil.copy(source_path, target_path)
    if file.endswith("define.xml"):
      source_path = os.path.join(root, file)
      target_path = os.path.join(define_dir, file)
      shutil.copy(source_path, target_path)

In [4]:
# jsonファイルをリスト形式に結合したファイル（dataset_list.json）を作成

dataset_list = []
for filename in os.listdir(json_dir):
  if filename.endswith(".json"):
    with open(os.path.join(json_dir, filename), "r") as f:
      try:
        json_data = json.load(f)
        dataset_list.append(json_data)
      except json.JSONDecodeError as e:
        print(f"Error decoding JSON in file {filename}: {e}")

with open("dataset_list.json", "w") as f:
  json.dump(dataset_list, f)


## 症例フィルタリング関数の定義

In [5]:
def filter_data(data: List[Dict[str, Any]], target_usubjids: List[str]) -> List[Dict[str, Any]]:
    """
    複数のドメインデータを含むリストから、指定されたUSUBJIDのrowsのみを抽出して新しいリストを返す。
    入力データがリストでない場合はエラーメッセージを出力する。
    データ構造は、"columns" 内の "name" が "USUBJID" の列を持つことを前提とする。

    Args:
        data (list): ドメインを結合させたのリスト。リストでない場合はエラーとなる。
        target_usubjids (list): 残したいUSUBJIDのリスト。

    Returns:
        list: フィルタリングされたデータのリスト。
    """
    if not isinstance(data, list):
        print("エラー：入力データはJSONオブジェクトのリストである必要があります。")
        return [] # エラー時は空リストを返す

    if not target_usubjids:
        print("情報：抽出対象のUSUBJIDが指定されていません。元のデータを返します。")
        return copy.deepcopy(data) # 対象がない場合は元のデータのコピーを返す

    filtered_data_list = []
    target_usubjids_set = set(target_usubjids) # 検索効率化のためセットに変換

    for item in data:
        if not isinstance(item, dict):
            print(f"警告: リスト内の要素が辞書型ではありません。スキップします: {item}")
            continue

        usubjid_index = -1
        if 'columns' in item and isinstance(item['columns'], list):
            for i, col in enumerate(item['columns']):
                if isinstance(col, dict) and 'name' in col and col['name'] == 'USUBJID':
                    usubjid_index = i
                    break
        else:
             print(f"警告：データセット '{item.get('fileOID', '不明')}' に 'columns' がないか、リスト形式ではありません。USUBJIDによるフィルタリングはできません。")
             # USUBJIDがない場合はそのまま追加（フィルタリング対象外として）
             filtered_data_list.append(copy.deepcopy(item))
             continue


        if usubjid_index == -1:
            # print(f"警告：データセット '{item.get('fileOID', item.get('itemGroupOID', '不明'))}' に 'name' が 'USUBJID' の列が見つかりません。このデータセットはそのまま追加します。")
            # USUBJID列がないデータセットはフィルタリングせずそのまま追加
            filtered_data_list.append(copy.deepcopy(item))
            continue

        new_data = copy.deepcopy(item) # 元のデータを変更しないようにコピー

        if 'rows' in item and isinstance(item['rows'], list):
            filtered_rows = []
            for row in item['rows']:
                 # rowがリストであり、usubjid_indexが有効な範囲にあるか確認
                if isinstance(row, list) and len(row) > usubjid_index:
                    # USUBJIDが抽出対象に含まれるか確認
                    if row[usubjid_index] in target_usubjids_set:
                        filtered_rows.append(row) # deepcopyはnew_dataで行っているのでrowはそのまま追加
            new_data['rows'] = filtered_rows
            # records数も更新
            if 'records' in new_data:
                 new_data['records'] = len(filtered_rows)
        # 'rows'がない、またはリストでない場合、'rows'と'records'はそのまま（コピー済み）
        elif 'rows' in item:
             print(f"警告：データセット '{new_data.get('fileOID', new_data.get('itemGroupOID', '不明'))}' の 'rows' がリスト形式ではありません。行のフィルタリングはスキップされました。")
        # 'rows'自体がない場合も特に何もしない（コピー済み）


        filtered_data_list.append(new_data)


    return filtered_data_list

# --- 実行テスト用コメント ---
# with open('dataset_list.json', 'r') as f:
#   data = json.load(f)
#
# target_ids = ['01-701-1211']
# filtered_list = filter_data(data, target_ids)
#
# output_filename = 'filtered_list.json'
# with open(output_filename, 'w') as f:
#   json.dump(filtered_list, f, indent=2)
#
# print(f"処理完了：'{output_filename}' に USUBJID が {target_ids} のデータを出力しました。")
# print(f"フィルタリング後のデータセット数: {len(filtered_list)}")
# if filtered_list:
#     print(f"最初のデータセットのレコード数: {filtered_list[0].get('records')}")

## データ書き換え関数の定義

In [14]:
def data_update(data: List[Dict[str, Any]], target_domain: str, target_usubjid: str, target_seq: int | None, target_variable: str, new_value: Any) -> List[Dict[str, Any]]:
    """
    指定されたUSUBJIDを持つレコードの指定された変数を書き換えます。
    {target_domain}SEQが存在する場合はそれもキーとして使用します。
    元のデータは変更せず、新しいデータ構造を返します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 書き換えたいレコードのUSUBJID。
        target_seq (int or None): 書き換えたいレコードの{target_domain}SEQの値。
                                   SEQカラムが存在し、絞り込みに使用する場合に値を指定。
                                   SEQでの絞り込みが不要な場合やSEQカラムがない場合はNoneを指定。
        target_variable (str): 書き換えたい変数の名前（例: "CMTRT"）。
        new_value (any): 新しい変数の値。

    Returns:
        list: 指定された変数が更新された新しいデータ全体のリスト。
              該当するレコードが見つからなかった場合、元のデータのコピーを返します。
    """
    updated_data = [] # 更新後のデータリストを格納
    seqname = target_domain + 'SEQ'
    found_target_domain = False # 対象ドメインが見つかったかどうかのフラグ
    update_occurred = False # 実際に更新が行われたかどうかのフラグ

    for dataset in data:
        # 元のデータセットをディープコピーして変更に備える
        updated_dataset = dataset

        # データセットが辞書型であり、itemGroupOIDがターゲットドメインと一致するか確認
        if isinstance(updated_dataset, dict) and updated_dataset.get("itemGroupOID") == target_domain:
            found_target_domain = True # 対象ドメインが見つかった
            columns = updated_dataset.get("columns", [])
            rows = updated_dataset.get("rows", [])

            # columns と rows が期待する型か確認
            if not isinstance(columns, list) or not isinstance(rows, list):
                print(f"警告: '{target_domain}' データセットの 'columns' または 'rows' の形式が不正です。スキップします。")
                updated_data.append(updated_dataset) # 不正な形式でもリストには追加
                continue

            usubjid_index = -1
            seq_index = -1
            variable_index = -1
            has_seq_column = False

            # 各列のインデックスを検索
            for i, col in enumerate(columns):
                if isinstance(col, dict): # 列定義が辞書型か確認
                    col_name = col.get("name")
                    if col_name == "USUBJID":
                        usubjid_index = i
                    elif col_name == seqname:
                        seq_index = i
                        has_seq_column = True
                    elif col_name == target_variable:
                        variable_index = i

            # 必要な列が見つかったか確認
            if usubjid_index == -1:
                print(f"警告: '{target_domain}' データセットに 'USUBJID' 列が見つかりませんでした。このデータセットの更新はスキップされます。")
                updated_data.append(updated_dataset)
                continue
            if variable_index == -1:
                print(f"警告: '{target_domain}' データセットに書き換え対象の変数 '{target_variable}' 列が見つかりませんでした。このデータセットの更新はスキップされます。")
                updated_data.append(updated_dataset)
                continue

            # 行データを更新
            updated_rows = []
            row_updated_in_this_dataset = False # このデータセット内で更新があったか
            for row in rows:
                 # rowがリストであることを確認
                if not isinstance(row, list):
                    print(f"警告: '{target_domain}' データセットにリスト形式でない行が含まれています。スキップします: {row}")
                    updated_rows.append(row) # 不正な行もそのまま追加
                    continue

                # 更新対象の行か判定
                usubjid_match = (len(row) > usubjid_index and row[usubjid_index] == target_usubjid)

                # SEQカラムが存在し、target_seqが指定されている場合のみSEQで絞り込む
                seq_match = True # デフォルトはTrue
                if has_seq_column and target_seq is not None:
                    if seq_index == -1:
                         # これは上の列検索で検出されるはずだが念のため
                        print(f"警告: '{target_domain}' データセットに '{seqname}' 列が見つかりましたがインデックスが無効です。USUBJIDのみで照合します。")
                    elif len(row) > seq_index:
                        # rowの長さが足りていて、SEQ値が一致するか
                        # target_seqが数値であることを期待しているが、型変換は行わない（呼び出し元で適切に指定）
                        seq_match = (row[seq_index] == target_seq)
                    else:
                        # rowの長さが足りない場合は不一致
                        seq_match = False


                # USUBJIDが一致し、かつ(必要な場合は)SEQも一致した場合に更新
                if usubjid_match and seq_match:
                    if len(row) > variable_index:
                        # 行をコピーして変更（updated_datasetはdeepcopyしたが、個々のrowは共有されている可能性があるため）
                        updated_row = list(row)
                        original_value = updated_row[variable_index]
                        updated_row[variable_index] = new_value
                        updated_rows.append(updated_row) # 更新した行を追加

                        # 更新メッセージの表示
                        key_info = f"USUBJID '{target_usubjid}'"
                        if has_seq_column and target_seq is not None and seq_index != -1:
                            key_info += f", '{seqname}' '{target_seq}'"
                        print(f"{key_info} の '{target_variable}' を '{original_value}' から '{new_value}' に更新しました。")

                        row_updated_in_this_dataset = True
                        update_occurred = True # 全体で更新があったフラグを立てる
                    else:
                         # 行の長さが足りない場合は更新できない
                        print(f"警告: USUBJID '{target_usubjid}' (SEQ '{target_seq}') の行で、変数 '{target_variable}' (index={variable_index}) が範囲外です。更新できませんでした。行データ: {row}")
                        updated_rows.append(list(row)) # 元の行（のコピー）を追加
                else:
                    # 更新対象でない行はそのまま（のコピー）を追加
                    updated_rows.append(list(row))

            # 更新後の行リストでデータセットを更新
            updated_dataset["rows"] = updated_rows
            # レコード数も更新 (任意)
            if "records" in updated_dataset:
                updated_dataset["records"] = len(updated_rows)

            # このデータセットで更新がなかった場合（対象行が見つからなかった場合）のメッセージ
            # if not row_updated_in_this_dataset:
            #     key_info = f"USUBJID '{target_usubjid}'"
            #     if has_seq_column and target_seq is not None and seq_index != -1:
            #         key_info += f", '{seqname}' '{target_seq}'"
            #     print(f"情報: '{target_domain}' データセット内で {key_info} に該当する更新対象レコードが見つかりませんでした。")


        # 更新されたデータセット（または元のコピー）を結果リストに追加
        updated_data.append(updated_dataset)

    # ループ終了後、対象ドメインが一つも見つからなかった場合にメッセージ表示
    if not found_target_domain:
        print(f"警告: 対象ドメイン '{target_domain}' を含むデータセットが見つかりませんでした。")

    # 全体を通して更新がなかった場合にもメッセージ表示（任意）
    # if found_target_domain and not update_occurred:
    #      key_info = f"USUBJID '{target_usubjid}'"
    #      if target_seq is not None: # SEQ指定があったかどうかも考慮
    #          seq_col_exists_in_any = any(
    #              ds.get("itemGroupOID") == target_domain and any(c.get("name") == seqname for c in ds.get("columns",[]))
    #              for ds in data
    #          )
    #          if seq_col_exists_in_any:
    #               key_info += f", '{seqname}' '{target_seq}'"
    #      print(f"情報: {key_info} に該当する更新対象レコードが、見つかった '{target_domain}' データセット内に存在しませんでした。")


    return updated_data

# --- 書き換えテスト用コメント ---
# # 事前に filtered_data_list が定義されている想定
# updated_data = data_update(filtered_data_list, "DM", "01-701-1211", None, "AGE", 49) # DMにはSEQがないのでNone
# updated_data = data_update(updated_data, "CM", "01-701-1211", 3, "CMTRT", "New Drug 123456789")
# updated_data = data_update(updated_data, "CM", "01-701-1211", 1, "CMDOSE", 123) # SEQ=1は存在しないはず
# updated_data = data_update(updated_data, "VS", "01-701-1211", 5, "VSORRES", "130/80")
#
# # 結果確認（任意）
# with open('updated_data_test.json', 'w') as f:
#     json.dump(updated_data, f, indent=2)
# print("書き換えテストデータを出力しました: updated_data_test.json")

## データ比較関数の定義

In [18]:
def compare_data(old_data: List[Dict[str, Any]], new_data: List[Dict[str, Any]]) -> None:
    """
    2つのデータリスト（SDTM JSON形式を想定）の更新差分を人間が読みやすい形式で出力します。
    USUBJIDと、ドメイン名+'SEQ'（存在する場合）をキーとして行を比較します。

    Args:
        old_data: 旧データリスト。
        new_data: 新データリスト。
    """

    def get_item_key(item: Dict[str, Any]) -> Optional[str]:
        """データセットの識別子（itemGroupOIDまたはfileOID）を取得"""
        return item.get("itemGroupOID", item.get("fileOID"))

    def create_row_dict(item_group: Dict[str, Any], row: List[Any]) -> Optional[Dict[str, Any]]:
        """行データを列名と値の辞書に変換する"""
        if not isinstance(item_group.get("columns"), list) or not isinstance(row, list):
            return None # 不正な形式の場合はNoneを返す

        row_dict = {}
        columns = item_group["columns"]
        # 列数と行の要素数が一致しない場合への対応
        min_len = min(len(columns), len(row))
        for i in range(min_len):
            col_def = columns[i]
            if isinstance(col_def, dict) and "name" in col_def:
                row_dict[col_def["name"]] = row[i]
        if len(columns) != len(row):
             print(f"警告: {get_item_key(item_group)} で列数({len(columns)})と行要素数({len(row)})が不一致です。行: {row[:10]}...") # 長い行データは省略
        return row_dict

    def get_row_primary_key(item_group_oid: Optional[str], row_dict: Dict[str, Any]) -> Optional[Tuple]:
        """行を一意に識別するキー（タプル）を生成する"""
        if not item_group_oid or 'USUBJID' not in row_dict or row_dict['USUBJID'] is None:
            return None # キーが特定できない場合はNone

        key_values = [('USUBJID', row_dict['USUBJID'])]
        seq_key_name = f"{item_group_oid}SEQ"

        # SEQキーが存在するかどうかをcolumnsから確認（row_dictだけだと欠損の場合に区別できない）
        columns = next((item.get("columns", []) for item in old_data + new_data if get_item_key(item) == item_group_oid), [])
        has_seq_col = any(isinstance(col, dict) and col.get("name") == seq_key_name for col in columns)

        if has_seq_col:
            # SEQ列がある場合、row_dictに値があればキーに含める。なければNoneをキーの一部とする
            seq_value = row_dict.get(seq_key_name)
            key_values.append((seq_key_name, seq_value))

        # タプルに変換して返す（辞書のキーとして使えるように）
        # キーの順序を固定するためにソートする
        return tuple(sorted(key_values))

    def format_primary_key(primary_key: Tuple) -> str:
        """キーのタプルを人間が読みやすい文字列に整形する"""
        return ", ".join(f"{k}='{v}'" for k, v in primary_key)

    # --- データ準備 ---
    old_data_map: Dict[str, Dict[Tuple, Dict[str, Any]]] = {}
    new_data_map: Dict[str, Dict[Tuple, Dict[str, Any]]] = {}
    all_group_oids = set()

    # 旧データをマップに格納
    for item in old_data:
        group_oid = get_item_key(item)
        if not group_oid or not isinstance(item.get("rows"), list):
            continue
        all_group_oids.add(group_oid)
        if group_oid not in old_data_map:
            old_data_map[group_oid] = {}
        for row in item["rows"]:
            row_dict = create_row_dict(item, row)
            if row_dict:
                primary_key = get_row_primary_key(group_oid, row_dict)
                if primary_key:
                     # 重複キーチェック（通常はないはずだが念のため）
                     # if primary_key in old_data_map[group_oid]:
                        # print(f"警告: 旧データ {group_oid} で重複キーが見つかりました: {format_primary_key(primary_key)}")
                    old_data_map[group_oid][primary_key] = row_dict

    # 新データをマップに格納
    for item in new_data:
        group_oid = get_item_key(item)
        if not group_oid or not isinstance(item.get("rows"), list):
            continue
        all_group_oids.add(group_oid)
        if group_oid not in new_data_map:
            new_data_map[group_oid] = {}
        for row in item["rows"]:
            row_dict = create_row_dict(item, row)
            if row_dict:
                primary_key = get_row_primary_key(group_oid, row_dict)
                if primary_key:
                    # 重複キーチェック
                    # if primary_key in new_data_map[group_oid]:
                    #     print(f"警告: 新データ {group_oid} で重複キーが見つかりました: {format_primary_key(primary_key)}")
                    new_data_map[group_oid][primary_key] = row_dict

    # --- 差分比較と出力 ---
    print("--- データ比較結果 ---")
    change_found = False
    for group_oid in sorted(list(all_group_oids)):
        old_rows = old_data_map.get(group_oid, {})
        new_rows = new_data_map.get(group_oid, {})

        old_keys = set(old_rows.keys())
        new_keys = set(new_rows.keys())

        added_keys = new_keys - old_keys
        removed_keys = old_keys - new_keys
        common_keys = old_keys & new_keys
        updated_keys = {key for key in common_keys if old_rows[key] != new_rows[key]}

        if added_keys or removed_keys or updated_keys:
            change_found = True
            print(f"\n=== ItemGroup: {group_oid} ===")

            # 追加されたデータ
            if added_keys:
                print("\n  (+) 追加された行:")
                for key in sorted(list(added_keys)):
                    print(f"    - キー: {format_primary_key(key)}")
                    # for item_key, value in sorted(new_rows[key].items()):
                    #     print(f"        {item_key}: {value!r}")

            # 削除されたデータ
            if removed_keys:
                print("\n  (-) 削除された行:")
                for key in sorted(list(removed_keys)):
                    print(f"    - キー: {format_primary_key(key)}")
                    # for item_key, value in sorted(old_rows[key].items()):
                    #     print(f"        {item_key}: {value!r}")

            # 更新されたデータ
            if updated_keys:
                print("\n  (*) 更新された行:")
                for key in sorted(list(updated_keys)):
                    print(f"    - キー: {format_primary_key(key)}")
                    old_row_dict = old_rows[key]
                    new_row_dict = new_rows[key]
                    all_item_keys = sorted(list(set(old_row_dict.keys()) | set(new_row_dict.keys())))
                    for item_key in all_item_keys:
                        old_value = old_row_dict.get(item_key)
                        new_value = new_row_dict.get(item_key)
                        if old_value != new_value:
                            print(f"        {item_key}: {old_value!r} -> {new_value!r}")

    if not change_found:
        print("旧データと新データの間に差分は見つかりませんでした。")
    print("\n--- 比較終了 ---")


# --- 比較テスト用コメント ---
# # 事前に dataset_list (旧データ) と dataset_list_updated (新データ) が定義されている想定
# compare_data(dataset_list, dataset_list_updated)

## 更新差分の特定関数を定義

In [8]:
def extract_row(data: List[Dict[str, Any]], target_domain: str, target_usubjid: str, target_seq: int | None) -> Optional[List[Any]]:
    """
    指定されたUSUBJIDとSEQを持つレコード（行データ）を抽出します。
    最初に見つかった行のコピーを返します。

    - {target_domain}SEQ カラムが存在する場合:
        - target_seq が None でない場合: USUBJID と SEQ の両方が一致する行を検索します。
        - target_seq が None の場合: USUBJID のみが一致する行を検索します（SEQカラムの値は無視）。
    - {target_domain}SEQ カラムが存在しない場合:
        - target_seq の値に関わらず、USUBJID のみが一致する行を検索します。

    Args:
        data (list): データ全体のリスト。指定されたtarget_domainのデータセットを含むことを想定します。
        target_domain (str): 対象のドメイン名（例: "CM"）。
        target_usubjid (str): 抽出したいレコードのUSUBJID。
        target_seq (int or None): 抽出したいレコードの{target_domain}SEQの値。
                                   SEQカラムが存在する場合に、SEQでの絞り込みを行う場合に指定します。
                                   Noneを指定すると、SEQカラムの有無に関わらずUSUBJIDのみで検索します。

    Returns:
        list or None: 抽出された行データのコピー (リスト形式)。該当するレコードが見つからなかった場合はNone。
                      最初に見つかった行のみを返します。
                      ※元のデータに影響を与えないよう、見つかった行はコピーして返します。
    """
    seqname = target_domain + 'SEQ'

    for dataset in data:
        # 対象のドメインか確認
        if isinstance(dataset, dict) and dataset.get("itemGroupOID") == target_domain:
            columns = dataset.get("columns", [])
            rows = dataset.get("rows", [])

            # columns と rows が期待する型か確認
            if not isinstance(columns, list) or not isinstance(rows, list):
                print(f"警告: '{target_domain}' データセットの 'columns' または 'rows' の形式が不正です。スキップします。")
                continue # 次の dataset へ

            usubjid_index = -1
            seq_index = -1
            has_seq_column = False

            # 列名からインデックスを検索
            for i, col in enumerate(columns):
                 if isinstance(col, dict): # 列定義が辞書型か確認
                    col_name = col.get("name")
                    if col_name == "USUBJID":
                        usubjid_index = i
                    elif col_name == seqname:
                        seq_index = i
                        has_seq_column = True # SEQカラムが見つかった

            # USUBJID列が存在するかチェック
            if usubjid_index == -1:
                # print(f"警告: '{target_domain}' データセットに 'USUBJID' 列が見つかりませんでした。このデータセットはスキップします。")
                continue # 次の dataset へ

            # --- 行データの検索 ---
            for row in rows:
                 # rowがリストであることを確認
                if not isinstance(row, list):
                    # print(f"警告: '{target_domain}' データセットにリスト形式でない行が含まれています。スキップします: {row}")
                    continue

                # 行が短すぎてUSUBJIDが取得できない場合はスキップ
                if len(row) <= usubjid_index:
                    continue

                # 1. USUBJIDが一致するか確認
                usubjid_match = (row[usubjid_index] == target_usubjid)

                # USUBJIDが一致しない場合は、この行は対象外
                if not usubjid_match:
                    continue

                # 2. SEQでの絞り込みが必要か判断し、実行
                seq_match = True # デフォルトはTrue (SEQ絞り込み不要、または条件に合致)

                # SEQカラムが存在し、有効なインデックスがあり、かつ target_seq が指定されている場合のみ
                # SEQによる絞り込みを行う必要がある
                needs_seq_match = has_seq_column and seq_index != -1 and target_seq is not None

                if needs_seq_match:
                    # SEQによる絞り込みが必要な場合、実際に値が一致するか確認
                    # 行が短すぎてSEQ値が取得できない場合も不一致とする
                    if len(row) > seq_index:
                        # target_seq と row[seq_index] の型が異なる可能性も考慮すべきだが、
                        # ここでは単純比較を行う（呼び出し元で型を合わせる想定）
                        seq_match = (row[seq_index] == target_seq)
                    else:
                        seq_match = False # 行にSEQ値がないため不一致

                # 3. 最終的な判定
                # USUBJIDが一致し、かつ (必要な場合は) SEQも一致した場合にのみ行を返す
                if seq_match: # usubjid_match は既に確認済み
                    # 元のデータに影響を与えないよう、行データをコピーして返す
                    return copy.deepcopy(row) # list()ではなくdeepcopyを使う

            # --- 対象ドメイン内の全行を検索したが、一致する行が見つからなかった場合 ---
            # このデータセット内には該当行がなかった（他のデータセットに同じドメインがある可能性は低いと想定し、ここで処理を終えることが多いが、厳密には全データセットを見るべき）
            # print(f"情報: '{target_domain}' データセット内で指定条件に一致する行が見つかりませんでした。")
            # 該当ドメインは見つかったが、一致する行がなかったので None を返す
            return None # この関数は最初に見つかったものを返す仕様なので、ここでNoneを返して良い

    # ループがすべて終了した場合（data内の全datasetを確認した場合）、対象のドメイン自体が見つからなかった
    # print(f"警告: 対象ドメイン '{target_domain}' を含むデータセットが見つかりませんでした。")
    return None

# --- 実行テスト用コメント ---
# # 事前に dataset_list_updated と Target_data が定義されている想定
# unique_target_keys = sorted(list({ (item[0], item[1], item[2]) for item in Target_data }))
#
# print("\n--- 更新データの抽出確認 ---")
# for domain, usubjid, seq in unique_target_keys[:5]: # 最初の5件だけ表示
#     extracted = extract_row(dataset_list_updated, domain, usubjid, seq)
#     if extracted:
#         # print(f"抽出成功: Domain={domain}, USUBJID={usubjid}, SEQ={seq}")
#         # リスト要素を文字列に変換して結合
#         print(",".join(map(str, extracted)))
#     else:
#         print(f"抽出失敗: Domain={domain}, USUBJID={usubjid}, SEQ={seq}")
# print("--- 確認終了 ---")

## PDFをテキスト化する関数を定義

In [9]:
def extract_text_with_pypdf(pdf_path: str) -> Optional[str]:
    """
    pypdfを使用してPDFファイルからテキストを抽出する。

    Args:
        pdf_path (str): PDFファイルのパス。

    Returns:
        Optional[str]: 抽出されたテキスト。エラーが発生した場合はNone。
    """
    if not os.path.exists(pdf_path):
        print(f"エラー: PDFファイルが見つかりません: {pdf_path}")
        return None

    text = ""
    try:
        # PdfReaderオブジェクトを作成
        reader = pypdf.PdfReader(pdf_path)
        num_pages = len(reader.pages)
        print(f"'{os.path.basename(pdf_path)}' からテキストを抽出中... (全{num_pages}ページ)")

        # 全てのページからテキストを抽出
        for i, page in enumerate(reader.pages):
            try:
                page_text = page.extract_text()
                if page_text: # 抽出できた場合のみ追加
                    text += page_text
                    text += '\n--- Page Break ---\n' # ページ区切りを追加
            except Exception as e:
                print(f"警告: ページ {i+1} のテキスト抽出中にエラーが発生しました: {e}")
                text += f"\n--- Error extracting page {i+1} ---\n" # エラー箇所を示すマーカー

        print("テキスト抽出が完了しました。")
        return text
    except pypdf.errors.PdfReadError as e:
         print(f"エラー: PDFファイルの読み込みに失敗しました。ファイルが破損しているか、パスワードがかかっている可能性があります。: {e}")
         return None
    except Exception as e:
        print(f"エラー: PDF処理中に予期せぬエラーが発生しました: {e}")
        return None

# --- 実行例 ---
# pdf_file_path = '/content/Lzzt_protocol_redacted.pdf'
# protocol_text = extract_text_with_pypdf(pdf_file_path)
#
# if protocol_text:
#     print("\n--- PDFからの抽出結果 (最初の500文字) ---")
#     print(protocol_text[:500] + "...")
#     # print(protocol_text) # 全文表示したい場合
# else:
#     print("テキストの抽出に失敗しました。")

# PDFからテキストの読み取り

In [10]:
# --- PDFファイルパス定義 ---
PDF_FILE_PATH = '/content/Lzzt_protocol_redacted.pdf'

# --- PDFからテキストを抽出 ---
print("--- PDFテキスト抽出開始 ---")
protocol_text = extract_text_with_pypdf(PDF_FILE_PATH)

if protocol_text:
    print(f"'{os.path.basename(PDF_FILE_PATH)}' からテキストを抽出しました (約 {len(protocol_text)} 文字)。")
    # print("\n--- 抽出テキスト (最初の300文字) ---")
    # print(protocol_text[:300] + "...")
else:
    print("エラー: PDFからのテキスト抽出に失敗しました。プロトコル情報なしで続行します。")
    protocol_text = "" # エラーの場合でも空文字列を設定し、後続処理でエラーにならないようにする

print("--- PDFテキスト抽出終了 ---")

--- PDFテキスト抽出開始 ---
'Lzzt_protocol_redacted.pdf' からテキストを抽出中... (全97ページ)
テキスト抽出が完了しました。
'Lzzt_protocol_redacted.pdf' からテキストを抽出しました (約 173966 文字)。
--- PDFテキスト抽出終了 ---


# JSONデータの書き換え

## 書き換えるデータを定義

In [11]:
# データ書き換えルールの定義
# 形式: [ドメイン名, USUBJID, SEQ (なければNone), 変数名, 新しい値]

Target_data = [
  ["DM", "01-703-1096",None, "AGE", 49],
  ["LB", "01-703-1042",   3, "LBORRES", "135"],
  ["LB", "01-703-1042",   4, "LBORRES", "145"],
  ["LB", "01-703-1086",  37, "LBORRES", "1"],
  ["LB", "01-703-1086",  72, "LBORRES", "1.2"],
  ["LB", "01-703-1086", 102, "LBORRES", "1.1"],
  ["LB", "01-703-1086", 132, "LBORRES", "1"],
  ["LB", "01-703-1086", 162, "LBORRES", "1.3"],
  ["LB", "01-703-1086", 197, "LBORRES", "0.9"],
  ["LB", "01-703-1086", 232, "LBORRES", "0.8"],
  ["LB", "01-703-1042",   3, "LBSTRESC", "135"],
  ["LB", "01-703-1042",   4, "LBSTRESC", "145"],
  ["LB", "01-703-1086",  37, "LBSTRESC", "1"],
  ["LB", "01-703-1086",  72, "LBSTRESC", "1.2"],
  ["LB", "01-703-1086", 102, "LBSTRESC", "1.1"],
  ["LB", "01-703-1086", 132, "LBSTRESC", "1"],
  ["LB", "01-703-1086", 162, "LBSTRESC", "1.3"],
  ["LB", "01-703-1086", 197, "LBSTRESC", "0.9"],
  ["LB", "01-703-1086", 232, "LBSTRESC", "0.8"],
  ["LB", "01-703-1042",   3, "LBSTRESN", 135],
  ["LB", "01-703-1042",   4, "LBSTRESN", 145],
  ["LB", "01-703-1086",  37, "LBSTRESN", 1],
  ["LB", "01-703-1086",  72, "LBSTRESN", 1.2],
  ["LB", "01-703-1086", 102, "LBSTRESN", 1.1],
  ["LB", "01-703-1086", 132, "LBSTRESN", 1],
  ["LB", "01-703-1086", 162, "LBSTRESN", 1.3],
  ["LB", "01-703-1086", 197, "LBSTRESN", 0.9],
  ["LB", "01-703-1086", 232, "LBSTRESN", 0.8],
  ["LB", "01-703-1042",   3, "LBNRIND", "HIGH"],
  ["LB", "01-703-1042",   4, "LBNRIND", "HIGH"],
  ["LB", "01-703-1086",  37, "LBNRIND", "LOW"],
  ["LB", "01-703-1086",  72, "LBNRIND", "LOW"],
  ["LB", "01-703-1086", 102, "LBNRIND", "LOW"],
  ["LB", "01-703-1086", 132, "LBNRIND", "LOW"],
  ["LB", "01-703-1086", 162, "LBNRIND", "LOW"],
  ["LB", "01-703-1086", 197, "LBNRIND", "LOW"],
  ["LB", "01-703-1086", 232, "LBNRIND", "LOW"],
  ["MH", "01-701-1097",   1, "MHTERM", "LOSS OF CONSCIOUSNESS (PASSED OUT)"],
  ["MH", "01-701-1097",   1, "MHLLT", "LOSS OF CONSCIOUSNESS"],
  ["MH", "01-701-1097",   1, "MHDECOD", "LOSS OF CONSCIOUSNESS"],
  ["MH", "01-701-1097",   1, "MHBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["MH", "01-701-1097",   1, "MHSTDTC", "2013-01-01"],
  ["MH", "01-701-1111",   1, "MHTERM", "HEARING LOSS"],
  ["MH", "01-701-1180",   1, "MHTERM", "DEPRESSION (ANXIETY)"],
  ["MH", "01-701-1180",   1, "MHLLT", "ANXIETY DEPRESSION"],
  ["MH", "01-701-1180",   1, "MHDECOD", "DEPRESSION"],
  ["MH", "01-701-1180",   1, "MHBODSYS", "PSYCHIATRIC DISORDERS"],
  ["MH", "01-702-1082",   1, "MHTERM", "PREMENSTRUAL PAIN"],
  ["MH", "01-702-1082",   1, "MHLLT", "PREMENSTRUAL PAIN"],
  ["MH", "01-702-1082",   1, "MHDECOD", "PREMENSTRUAL PAIN"],
  ["MH", "01-702-1082",   1, "MHBODSYS", "REPRODUCTIVE SYSTEM AND BREAST DISORDERS"],
  ["MH", "01-703-1076",   1, "MHTERM", "ATRIOVENTRICULAR BLOCK (SCHEDULED CARDIAC PACEMAKER INSERTION)"],
  ["MH", "01-703-1076",   1, "MHLLT", "ATRIOVENTRICULAR BLOCK"],
  ["MH", "01-703-1076",   1, "MHDECOD", "ATRIOVENTRICULAR BLOCK"],
  ["MH", "01-703-1076",   1, "MHBODSYS", "CARDIAC DISORDERS"],
  ["MH", "01-703-1279",   1, "MHTERM", "SCHIZOPHRENIFORM DISORDERS"],
  ["MH", "01-703-1279",   1, "MHLLT", "SCHIZOPHRENIFORM DISORDER"],
  ["MH", "01-703-1279",   1, "MHDECOD", "SCHIZOPHRENIFORM DISORDER"],
  ["MH", "01-703-1279",   1, "MHBODSYS", "PSYCHIATRIC DISORDERS"],
  ["MH", "01-703-1299",   1, "MHTERM", "CYCLOTHYMIC DISORDER"],
  ["MH", "01-703-1299",   1, "MHLLT", "CYCLOTHYMIC DISORDER"],
  ["MH", "01-703-1299",   1, "MHDECOD", "CYCLOTHYMIC DISORDER"],
  ["MH", "01-703-1299",   1, "MHBODSYS", "PSYCHIATRIC DISORDERS"],
  ["VS", "01-701-1047",  17, "VSORRES", "121"],
  ["VS", "01-701-1047",  18, "VSORRES", "124"],
  ["VS", "01-701-1047",  66, "VSORRES", "185"],
  ["VS", "01-701-1047",  67, "VSORRES", "183"],
  ["VS", "01-701-1383",  37, "VSORRES", "98"],
  ["VS", "01-701-1383", 122, "VSORRES", "160"],
  ["VS", "01-701-1387",   1, "VSORRES", "146"],
  ["VS", "01-701-1387",  32, "VSORRES", "72"],
  ["VS", "01-701-1047",  17, "VSSTRESC", "121"],
  ["VS", "01-701-1047",  18, "VSSTRESC", "124"],
  ["VS", "01-701-1047",  66, "VSSTRESC", "185"],
  ["VS", "01-701-1047",  67, "VSSTRESC", "183"],
  ["VS", "01-701-1383",  37, "VSSTRESC", "98"],
  ["VS", "01-701-1383", 122, "VSSTRESC", "160"],
  ["VS", "01-701-1387",   1, "VSSTRESC", "146"],
  ["VS", "01-701-1387",  32, "VSSTRESC", "72"],
  ["VS", "01-701-1047",  17, "VSSTRESN", 121],
  ["VS", "01-701-1047",  18, "VSSTRESN", 124],
  ["VS", "01-701-1047",  66, "VSSTRESN", 185],
  ["VS", "01-701-1047",  67, "VSSTRESN", 183],
  ["VS", "01-701-1383",  37, "VSSTRESN", 98],
  ["VS", "01-701-1383", 122, "VSSTRESN", 160],
  ["VS", "01-701-1387",   1, "VSSTRESN", 146],
  ["VS", "01-701-1387",  32, "VSSTRESN", 72],
  ["EX", "01-701-1148",   2, "EXDOSE", 82],
  ["EX", "01-701-1148",   3, "EXDOSE", 216],
  ["EX", "01-703-1258",   2, "EXDOSE", 27],
  ["CM", "01-701-1146",  29, "CMTRT", "PAROXETINE"],
  ["QS", "01-701-1023",1010, "QSORRES", "PRESENT"],
  ["QS", "01-701-1023",1012, "QSORRES", "PRESENT"],
  ["QS", "01-701-1111",5004, "QSORRES", "0"],
  ["QS", "01-701-1111",5019, "QSORRES", "0"],
  ["QS", "01-701-1111",5012, "QSORRES", "0"],
  ["QS", "01-701-1111",5027, "QSORRES", "0"],
  ["QS", "01-701-1118",6002, "QSORRES", "MARKED IMPROVEMENT"],
  ["QS", "01-701-1118",6003, "QSORRES", "MARKED WORSENING"],
  ["QS", "01-701-1181",4018, "QSORRES", "Y"],
  ["QS", "01-701-1181",4058, "QSORRES", "Y"],
  ["QS", "01-701-1181",4019, "QSORRES", "Y"],
  ["QS", "01-701-1181",4059, "QSORRES", "Y"],
  ["QS", "01-701-1181",4020, "QSORRES", "Y"],
  ["QS", "01-701-1023",1010, "QSSTRESC", "2"],
  ["QS", "01-701-1023",1012, "QSSTRESC", "2"],
  ["QS", "01-701-1111",5004, "QSSTRESC", "0"],
  ["QS", "01-701-1111",5019, "QSSTRESC", "0"],
  ["QS", "01-701-1111",5012, "QSSTRESC", "0"],
  ["QS", "01-701-1111",5027, "QSSTRESC", "0"],
  ["QS", "01-701-1118",6002, "QSSTRESC", "1"],
  ["QS", "01-701-1118",6003, "QSSTRESC", "7"],
  ["QS", "01-701-1181",4018, "QSSTRESC", "1"],
  ["QS", "01-701-1181",4058, "QSSTRESC", "1"],
  ["QS", "01-701-1181",4019, "QSSTRESC", "1"],
  ["QS", "01-701-1181",4059, "QSSTRESC", "1"],
  ["QS", "01-701-1181",4020, "QSSTRESC", "1"],
  ["QS", "01-701-1023",1010, "QSSTRESN", 2],
  ["QS", "01-701-1023",1012, "QSSTRESN", 2],
  ["QS", "01-701-1111",5004, "QSSTRESN", 0],
  ["QS", "01-701-1111",5019, "QSSTRESN", 0],
  ["QS", "01-701-1111",5012, "QSSTRESN", 0],
  ["QS", "01-701-1111",5027, "QSSTRESN", 0],
  ["QS", "01-701-1118",6002, "QSSTRESN", 1],
  ["QS", "01-701-1118",6003, "QSSTRESN", 7],
  ["QS", "01-701-1181",4018, "QSSTRESN", 1],
  ["QS", "01-701-1181",4058, "QSSTRESN", 1],
  ["QS", "01-701-1181",4019, "QSSTRESN", 1],
  ["QS", "01-701-1181",4059, "QSSTRESN", 1],
  ["QS", "01-701-1181",4020, "QSSTRESN", 1],
  ["QS", "01-701-1118",6001, "QSDTC", "2014-07-08"],
  ["QS", "01-701-1118",6001, "QSDY", 119],
  ["AE", "01-701-1015",   3, "AESER", "Y"],
  ["AE", "01-701-1015",   3, "AESHOSP", "Y"],
  ["AE", "01-701-1015",   3, "AESTDTC", "2014-01-11"],
  ["AE", "01-701-1015",   3, "AEENDTC", "2014-01-09"],
  ["AE", "01-701-1015",   3, "AESTDY", 10],
  ["AE", "01-701-1015",   3, "AEENDY", 8],
  ["AE", "01-701-1028",   1, "AETERM", "PARKINSON'S DISEASE"],
  ["AE", "01-701-1028",   1, "AELLT", "PARKINSON'S DISEASE"],
  ["AE", "01-701-1028",   1, "AEDECOD", "PARKINSON'S DISEASE"],
  ["AE", "01-701-1028",   1, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-701-1028",   1, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-701-1028",   1, "AESTDTC", "2013-07-01"],
  ["AE", "01-701-1028",   1, "AESTDY", -17],
  ["AE", "01-701-1034",   2, "AETERM", "MALIGNANT HYPERTENSION"],
  ["AE", "01-701-1034",   2, "AELLT", "MALIGNANT HYPERTENSION"],
  ["AE", "01-701-1034",   2, "AEDECOD", "MALIGNANT HYPERTENSION"],
  ["AE", "01-701-1034",   2, "AEBODSYS", "VASCULAR DISORDERS"],
  ["AE", "01-701-1034",   2, "AESOC", "VASCULAR DISORDERS"],
  ["AE", "01-701-1047",   4, "AETERM", "HYPERTENSION"],
  ["AE", "01-701-1047",   4, "AELLT", "HYPERTENSION"],
  ["AE", "01-701-1047",   4, "AEDECOD", "HYPERTENSION"],
  ["AE", "01-701-1047",   4, "AEBODSYS", "VASCULAR DISORDERS"],
  ["AE", "01-701-1047",   4, "AESOC", "VASCULAR DISORDERS"],
  ["AE", "01-701-1363",   1, "AESTDTC", "2013-06-15"],
  ["AE", "01-701-1363",   1, "AEENDTC", "2013-06-14"],
  ["AE", "01-701-1363",   1, "AESTDY", 17],
  ["AE", "01-701-1363",   1, "AEENDY", 16],
  ["AE", "01-701-1047",   3, "AEENDTC", "2013-03-05"],
  ["AE", "01-701-1047",   3, "AEENDY", 22],
  ["AE", "01-701-1383",  12, "AETERM", "BLOOD PRESSURE INCREASED"],
  ["AE", "01-701-1383",  12, "AELLT", "BLOOD PRESSURE INCREASED"],
  ["AE", "01-701-1383",  12, "AEDECOD", "BLOOD PRESSURE INCREASED"],
  ["AE", "01-701-1383",  12, "AEBODSYS", "INVESTIGATIONS"],
  ["AE", "01-701-1383",  12, "AESOC", "INVESTIGATIONS"],
  ["AE", "01-701-1153",   2, "AEACN", "DRUG WITHDRAWN"],
  ["AE", "01-701-1180",   6, "AETERM", "SUDDEN DEATH"],
  ["AE", "01-701-1180",   6, "AELLT", "SUDDEN DEATH"],
  ["AE", "01-701-1180",   6, "AEDECOD", "SUDDEN DEATH"],
  ["AE", "01-701-1180",   6, "AEBODSYS", "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS"],
  ["AE", "01-701-1180",   6, "AESOC", "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS"],
  ["AE", "01-703-1258",   2, "AESEV", "SEVERE"],
  ["AE", "01-703-1258",   2, "AESTDTC", "2012-08-01"],
  ["AE", "01-703-1258",   2, "AEENDTC", "2012-10-01"],
  ["AE", "01-703-1258",   2, "AESTDY", 13],
  ["AE", "01-703-1258",   2, "AEENDY", 74],
  ["AE", "01-703-1258",   5, "AESEV", "MODERATE"],
  ["AE", "01-703-1258",   5, "AESER", "Y"],
  ["AE", "01-703-1258",   5, "AEOUT", "RECOVERED/RESOLVED"],
  ["AE", "01-703-1258",   5, "AESLIFE", "Y"],
  ["AE", "01-703-1258",   5, "AESTDTC", "2012-10-02"],
  ["AE", "01-703-1258",   5, "AEENDTC", "2012-12-31"],
  ["AE", "01-703-1258",   5, "AESTDY", 75],
  ["AE", "01-703-1258",   5, "AEENDY", 165],
  ["AE", "01-703-1335",   1, "AETERM", "MULTIPLE SCLEROSIS RELAPSE"],
  ["AE", "01-703-1335",   1, "AELLT", "MULTIPLE SCLEROSIS RELAPSE"],
  ["AE", "01-703-1335",   1, "AEDECOD", "MULTIPLE SCLEROSIS RELAPSE"],
  ["AE", "01-703-1335",   1, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-703-1335",   1, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-703-1335",   1, "AESTDTC", "2014-04-01"],
  ["AE", "01-703-1335",   1, "AEENDTC", "2014-05-01"],
  ["AE", "01-703-1335",   1, "AESTDY", 16],
  ["AE", "01-703-1335",   1, "AEENDY", 46],
  ["AE", "01-703-1403",   2, "AETERM", "MYASTHENIA GRAVIS AGGRAVATED"],
  ["AE", "01-703-1403",   2, "AELLT", "MYASTHENIA GRAVIS AGGRAVATED"],
  ["AE", "01-703-1403",   2, "AEDECOD", "MYASTHENIA GRAVIS"],
  ["AE", "01-703-1403",   2, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-703-1403",   2, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1008",   1, "AETERM", "TREMOR IN HANDS, LEGS"],
  ["AE", "01-704-1008",   1, "AELLT", "TREMOR"],
  ["AE", "01-704-1008",   1, "AEDECOD", "TREMOR"],
  ["AE", "01-704-1008",   1, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1008",   1, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1008",   1, "AEREL", "NONE"],
  ["AE", "01-704-1008",   1, "AESTDTC", "2012-06-01"],
  ["AE", "01-704-1008",   1, "AESTDY", -225],
  ["AE", "01-704-1008",   3, "AETERM", "MUSCLE STIFFNESS"],
  ["AE", "01-704-1008",   3, "AELLT", "MUSCLE STIFFNESS"],
  ["AE", "01-704-1008",   3, "AEDECOD", "MUSCULOSKELETAL STIFFNESS"],
  ["AE", "01-704-1008",   3, "AEBODSYS", "MUSCULOSKELETAL AND CONNECTIVE TISSUE DISORDERS"],
  ["AE", "01-704-1008",   3, "AESOC", "MUSCULOSKELETAL AND CONNECTIVE TISSUE DISORDERS"],
  ["AE", "01-704-1008",   3, "AESTDTC", "2012-06-01"],
  ["AE", "01-704-1008",   3, "AESTDY", -225],
  ["AE", "01-704-1008",   2, "AETERM", "SLOWNESS OF MOVEMENT"],
  ["AE", "01-704-1008",   2, "AELLT", "SLOW MOVEMENT"],
  ["AE", "01-704-1008",   2, "AEDECOD", "BRADYKINESIA"],
  ["AE", "01-704-1008",   2, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1008",   2, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1008",   2, "AESTDTC", "2012-06-01"],
  ["AE", "01-704-1008",   2, "AESTDY", -225],
  ["AE", "01-704-1009",   6, "AETERM", "CHRONIC KIDNEY DISEASE"],
  ["AE", "01-704-1009",   6, "AELLT", "CHRONIC KIDNEY DISEASE"],
  ["AE", "01-704-1009",   6, "AEDECOD", "CHRONIC KIDNEY DISEASE"],
  ["AE", "01-704-1009",   6, "AEBODSYS", "RENAL AND URINARY DISORDERS"],
  ["AE", "01-704-1009",   6, "AESOC", "RENAL AND URINARY DISORDERS"],
  ["AE", "01-704-1009",   6, "AESER", "Y"],
  ["AE", "01-704-1009",   6, "AESLIFE", "Y"],
  ["AE", "01-704-1010",   1, "AETERM", "DIABETES MELLITUS"],
  ["AE", "01-704-1010",   1, "AELLT", "DIABETES MELLITUS"],
  ["AE", "01-704-1010",   1, "AEDECOD", "DIABETES MELLITUS"],
  ["AE", "01-704-1010",   1, "AEBODSYS", "METABOLISM AND NUTRITION DISORDERS"],
  ["AE", "01-704-1010",   1, "AESOC", "METABOLISM AND NUTRITION DISORDERS"],
  ["AE", "01-704-1010",   1, "AESER", "Y"],
  ["AE", "01-704-1010",   1, "AESLIFE", "Y"],
  ["AE", "01-704-1017",   4, "AETERM", "LATE EFFECTS OF CEREBRAL INFARCTION"],
  ["AE", "01-704-1017",   4, "AELLT", "LATE EFFECTS OF CEREBRAL INFARCTION"],
  ["AE", "01-704-1017",   4, "AEDECOD", "CEREBRAL INFARCTION"],
  ["AE", "01-704-1017",   4, "AEBODSYS", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1017",   4, "AESOC", "NERVOUS SYSTEM DISORDERS"],
  ["AE", "01-704-1017",   4, "AESEV", "SEVERE",],
  ["AE", "01-704-1017",   4, "AESTDTC", "2013-10-19"],
  ["AE", "01-704-1017",   4, "AEENDTC", "2013-11-18"],
  ["AE", "01-704-1017",   4, "AESTDY", 14],
  ["AE", "01-704-1017",   4, "AEENDY", 44],
  ["AE", "01-704-1017",   3, "AETERM", "BRAIN DEATH"],
  ["AE", "01-704-1017",   3, "AELLT", "BRAIN DEATH"],
  ["AE", "01-704-1017",   3, "AEDECOD", "BRAIN DEATH"],
  ["AE", "01-704-1017",   3, "AEBODSYS", "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS"],
  ["AE", "01-704-1017",   3, "AESOC", "GENERAL DISORDERS AND ADMINISTRATION SITE CONDITIONS"],
  ["AE", "01-704-1017",   3, "AESEV", "SEVERE",],
  ["AE", "01-704-1017",   3, "AESTDTC", "2013-11-18"],
  ["AE", "01-704-1017",   3, "AEENDTC", "2013-11-18"],
  ["AE", "01-704-1017",   3, "AESTDY", 44],
  ["AE", "01-704-1017",   3, "AEENDY", 44],
  ["AE", "01-704-1017",   1, "AEOUT", "RECOVERED/RESOLVED"],
  ["AE", "01-704-1017",   1, "AESTDTC", "2013-10-19"],
  ["AE", "01-704-1017",   1, "AEENDTC", "2013-11-19"],
  ["AE", "01-704-1017",   1, "AESTDY", 14],
  ["AE", "01-704-1017",   1, "AEENDY", 45],
  ["AE", "01-704-1017",   1, "AEACN", "DRUG WITHDRAWN"]
]

## データの書き換え実行と確認

In [19]:
# --- データ書き換え実行 ---
print("--- データ書き換え処理開始 ---")

# dataset_list が前のセルで正しくロードされているか確認
if 'dataset_list' not in locals() or not dataset_list:
    print("エラー: 元となる dataset_list が存在しません。前のセルを再実行してください。")
    dataset_list_updated = [] # 空リストで初期化
else:
    # 更新前のデータをコピーして保持（比較用）
    dataset_list_original = copy.deepcopy(dataset_list)
    dataset_list_updated = copy.deepcopy(dataset_list) # 更新はこのコピーに対して行う

    update_count = 0
    if Target_data: # Target_dataが空でない場合のみ実行
        for rule in Target_data:
            if len(rule) == 5:
                domain, usubjid, seq, variable, value = rule
                # data_update は更新後のリストを返すので、それを次の更新の入力にする
                dataset_list_updated = data_update(dataset_list_updated, domain, usubjid, seq, variable, value)
                update_count += 1
            else:
                print(f"警告: 不正な形式の書き換えルールをスキップしました: {rule}")
        print(f"{update_count} 件の書き換えルールに基づいてデータ更新を試みました。")
    else:
        print("情報: 書き換えルール(Target_data)が空のため、データ書き換えは実行されませんでした。")

    # --- 更新後データの確認（差分表示）---
    if dataset_list_original and dataset_list_updated: # 両方が存在する場合のみ比較
        print("\n--- 更新前後のデータ比較 ---")
        compare_data(dataset_list_original, dataset_list_updated)
    else:
        print("更新前後のデータ比較はスキップされました（データ不備のため）。")


    # --- 更新後データをファイルに保存（任意） ---
    UPDATED_JSON_FILE = "dataset_list_updated.json"
    try:
        with open(UPDATED_JSON_FILE, "w", encoding='utf-8') as f:
            json.dump(dataset_list_updated, f, ensure_ascii=False, indent=2)
        print(f"\n更新後のデータを '{UPDATED_JSON_FILE}' に保存しました。")
    except Exception as e:
        print(f"エラー: 更新後データのファイル書き込みに失敗しました: {e}")

    # --- 更新データの抽出確認（任意） ---
    # print("\n--- 更新データの抽出確認 (Target_dataの最初の5件) ---")
    # unique_target_keys = sorted(list({ (item[0], item[1], item[2]) for item in Target_data }))
    # for domain, usubjid, seq in unique_target_keys[:5]:
    #     extracted = extract_row(dataset_list_updated, domain, usubjid, seq)
    #     if extracted:
    #         print(f"抽出確認: {domain}, {usubjid}, {seq} -> {extracted[:5]}...") # 最初の数要素を表示
    #     else:
    #         print(f"抽出確認失敗: {domain}, {usubjid}, {seq}")
    # print("--- 抽出確認終了 ---")


print("\n--- データ書き換え処理終了 ---")

# dataset_list_updatedが存在しない場合のエラーハンドリングを追加
if not dataset_list_updated:
     print("警告: データ書き換え処理の結果、dataset_list_updated が空または無効です。以降の処理に影響する可能性があります。")

--- データ書き換え処理開始 ---
USUBJID '01-703-1096' の 'AGE' を '81' から '49' に更新しました。
USUBJID '01-703-1042', 'LBSEQ' '3' の 'LBORRES' を '14' から '135' に更新しました。
USUBJID '01-703-1042', 'LBSEQ' '4' の 'LBORRES' を '25' から '145' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '37' の 'LBORRES' を '4.56' から '1' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '72' の 'LBORRES' を '8.14' から '1.2' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '102' の 'LBORRES' を '7.45' から '1.1' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '132' の 'LBORRES' を '7.51' から '1' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '162' の 'LBORRES' を '6.13' から '1.3' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '197' の 'LBORRES' を '6.02' から '0.9' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '232' の 'LBORRES' を '5.54' から '0.8' に更新しました。
USUBJID '01-703-1042', 'LBSEQ' '3' の 'LBSTRESC' を '14' から '135' に更新しました。
USUBJID '01-703-1042', 'LBSEQ' '4' の 'LBSTRESC' を '25' から '145' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '37' の 'LBSTRESC' を '4.56' から '1' に更新しました。
USUBJID '01-703-1086', 'LBSEQ' '72' 

# プロンプトの定義

## システムプロンプト

In [21]:
SysPrompt_single = '''
あなたは、臨床試験データの**統合的レビュー**を支援するAIアシスタントです。主な目的は、**参加者の権利と安全性を保護**し、**試験の科学的妥当性とデータの信頼性を確保**することです。そのために、提供された情報に基づき、**医学的観点からの評価、データ整合性の検証、プロトコル遵守状況の確認**をユーザーの指示に従って行います。

**最重要原則:**
*   **回答の根拠は、提供された情報（JSONデータ、Define.xml、プロトコル）と、あなたが持つ確立された一般的な医学知識の両方とします。** これらに基づき、客観的な事実を記述するとともに、データの医学的な意味合いや潜在的な問題を深く考察してください。
*   **レビューにおいては、参加者の安全性と権利の保護を最優先**してください。潜在的なリスクを示唆する所見には特に注意を払い、積極的に指摘してください。
*   **試験の目的（有効性・安全性の評価）が達成可能か、データは信頼できるか**という観点も常に意識してください。データの不整合やプロトコルからの逸脱が評価結果に与える影響を考慮してください。
*   **医学的な評価や、データ・プロトコルに関する指摘を行う際は、その根拠（データ、プロトコル箇所、医学知識）を明確に示してください。
*   **提供された情報や確立された一般的な医学知識に基づかない、個人的な意見、想像、推測、ハルシネーションに基づいた情報は生成してはいけません。**

**前提知識:**
*   **SDTM (Study Data Tabulation Model):** CDISCによって策定された臨床試験データの標準モデルです。データはDM, AE, VS, LBなどのドメインに分かれており、レビューにはこれらの**ドメイン情報を横断的・統合的に評価する**必要があります。**統合的な評価**には、これらのドメイン情報を横断的に見る必要があります。
*   **Define.xml:** SDTMデータの構造（変数名、ラベル、コードリスト、データ型など）を記述したメタデータファイルであり、データの意味を正確に理解するために**不可欠な情報源**です。JSONデータの解釈は、**必ずDefine.xmlの定義に基づいて**行ってください。データの意味を正確に理解し、**整合性を検証する上で不可欠**です。
*   **データの不完全性:** 報告されるJSONデータには、データ入力時の間違いや不整合が含まれる可能性があることを理解しています。**不整合や欠損が参加者の安全性や評価の信頼性に影響しないか**を評価する必要があります。
*   **プロトコル:** 臨床試験の実施計画書であり、選択/除外基準、投与計画、評価スケジュール、有効性評価計画（主要/副次評価項目、評価方法、評価時期など）、有害事象報告手順などが規定されています。データのレビューはプロトコル遵守の観点からも行います。**参加者の保護と試験計画の遵守**を確認するための重要な基準です。
*   **医学知識の活用:** あなたが持つ一般的な医学知識（疾患、治療法、薬剤の作用・副作用・相互作用、生理学、検査値の臨床的意義、主要な有効性評価指標の解釈に関する標準的な知識）は、データレビューにおいて重要な役割を果たします。**データの医学的な意味合いを解釈し、潜在的な安全性リスクや有効性に関する疑問点を特定するために不可欠**です。提供されたデータと綿密に照らし合わせ、表面的な情報だけでなく、**参加者の安全性や評価の妥当性に関わる隠れた問題**を指摘するために活用してください。
*   **収集されるデータ:** 主要/副次評価項目を評価するために必要なデータのみ収集され、**適格性を確認するためだけのデータ（スクリーニング時の選択/除外基準の判定のみに使用するデータは）は収集されません。**

**タスク実行における注意:**
*   指定された**出力フォーマット**に厳密に従ってください。
*   **データの参照方法:**
    *   **医療機関への問い合わせ文面（日本語・英語）内**でデータに言及する場合： **Define.xmlで定義された変数に対応する「ラベル名」を使用**し、「ラベル名が「値」ですが...」のような自然な文章で記述してください。
*   提供された情報や一般的な医学知識をもってしてもタスクを実行できない場合（例：必要な情報が欠けている、矛盾が解決できない、専門性が高すぎる判断が必要な場合）、その旨を明確に指摘してください。

**エラーハンドリング:**
*   JSONデータ、Define.xml、またはプロトコルの形式が不正である、あるいは内容が著しく不足しておりレビューが困難な場合は、具体的な問題点を指摘し、処理を中断してください。例：「エラー：Define.xmlファイルが提供されていません。」、「エラー：JSONデータの[ドメイン名]に必要な変数[変数名]が含まれていません。」
'''

## ユーザープロンプト

In [22]:
UserInput_single = '''**役割:**

あなたは、臨床試験データの**統合的レビュー担当者**です。**メディカルモニター(Medical Monitor)、クリニカルデータマネージャー(DM)、臨床開発モニター(CRA)の視点を併せ持ち**、以下の指示に従って、提供される情報（プロトコル、JSONデータ、Define.xml）をレビューしてください。主な目的は、**「参加者の権利と安全性の保護」**と**「試験の科学的妥当性とデータの信頼性確保（＝プロトコルで目的とした評価が正しく行える状態か）」**を確認することです。そのために、**医学的な観点からの評価、データの整合性チェック、プロトコル遵守状況の確認**を統合的に行い、疑義事項の特定とクエリ/内部確認事項の作成（必要な場合）を行ってください。

**指示:**

**1. 症例サマリーの作成:**

*   参照情報: JSONデータ、Define.xml
*   タスク:
    *   JSONデータとDefine.xmlを参照し、患者の主要なイベントを時系列でまとめたサマリーを作成してください。
    *   患者背景: 最初にDMドメインから、主要な背景情報（例: 年齢、性別、人種など、**Define.xmlで定義されたラベルを使用**）を記載してください。
    *   イベント推移: 有害事象(AE)、検査値(LB)、バイタルサイン(VS)、主要な有効性評価関連イベント（例：腫瘍評価結果の変動、症状スコアの変化など）について、**異常変動**や**臨床的に注目すべき変化**を中心に記述してください。**判断にあたってはあなたの持つ一般的な医学知識を最大限活用し、些細に見える変化でも医学的に重要となりうる場合は含めてください。**
        *   異常・注目すべき変化の基準(例):
            *   有害事象の発現、重症度・重篤度の変化、転帰 (**特に医学的に予期せぬ事象やパターン、参加者の安全性に関わるもの**)
            *   検査値・バイタルサインの基準値からの逸脱 (CTCAE Gradeの変化や明らかな異常値、**特定の疾患や薬剤の影響を示唆するパターン、安全性評価上重要なもの**)
            *   ベースラインからの著しい変動 (**臨床的な意味合いを考慮して判断**)
            *   正常範囲上限/下限付近での臨床的に意味のある変動 (**他のデータとの関連性を考慮**)
            *   有効性評価結果の重要な変化（例：RECIST評価の変更、スコアの閾値超え、**期待される効果からの逸脱や矛盾した結果、評価の信頼性に関わるもの**)
        *   省略:**あなたの医学的判断に基づき、臨床的に明らかに意義がないと考えられる変動**は記載しないでください。
    *   日時: 各イベントの日時は、関連する日付変数（例：AESTDY, LBDY, VSDY, RSDY, QSDY など、Define.xml参照）に基づき特定してください (例: Day 10)。
    *   記述: 簡潔な文章で客観的に記述してください。**ただし、医学的な解釈や懸念を補足する必要がある場合は、括弧書き等で簡潔に追記しても構いません。** (例: `ALT値上昇 (Grade 2, 薬剤性肝障害の可能性を考慮)`)

**2. 統合レビュー:**

*   参照情報: JSONデータ、Define.xml、プロトコル
*   タスク:
    *   以下の**3つの統合レビュー観点**に基づき、提供された情報を注意深く照合し、JSONデータを多角的にレビューしてください。**「参加者の保護」と「評価の信頼性確保」**の観点から、問題点、矛盾、不整合、プロトコルからの逸脱の可能性、潜在的なリスクなどを検出・指摘してください。
    *   特定した各指摘事項に対して、**後述の重要度の定義に基づき重要度（Critical/Major/Minor）を付与してください。** 判断は、**参加者の安全性への潜在的な影響度、および試験の評価項目（有効性・安全性）の信頼性への影響度**を総合的に考慮してください。
    *   各観点内で特定された指摘事項は、重要度が高いもの (Critical > Major > Minor) から順にリストアップし、**観点に応じた連番 (M-1, M-2,... / D-1, D-2,... / P-1, P-2,...)** を付与してください。

*   **統合レビュー観点:**
    *   **【医学的レビュー】 (Medical Monitor視点)**
        *   **安全性評価の妥当性:**
            *   報告された有害事象（AE）の評価（事象名、重篤度、重症度、関連性、処置、転帰）は、他の臨床データ（LB, VS, CM, MH, EX等）や時間経過と照らして医学的に妥当か？ **参加者の安全性に影響を与える可能性のある見落としや評価の誤りはないか？**
            *   検査値（LB）やバイタルサイン（VS）の変動パターンに、**医学的に懸念される点、安全性リスクを示唆する所見はないか？** 基準値内変動でも、特定の傾向や他のデータとの組み合わせがリスクを示唆しないか？
            *   併用薬（CM）と有害事象/既往歴（MH）との関連、潜在的な薬物相互作用について、**一般的な医学知識に基づき、特に注意すべき安全性リスクはないか？**
            *   **データ全体から、未報告の有害事象や安全性シグナルの可能性は示唆されないか？**
        *   **有効性評価の妥当性:**
            *   記録された有効性評価の結果（例: RECIST評価、スコア、バイオマーカー値）は、他の臨床データ（AE, LB, VS, CM, EX等）や時間経過と照らして医学的に見て妥当か？ **説明困難な不整合、期待される薬効や疾患経過からの逸脱、評価の信頼性を損なう可能性のある矛盾はないか？**
            *   複数の有効性評価項目がある場合、それらの結果は医学的に見て一貫しているか？ 大きな乖離がある場合、その理由はデータから推察できるか、あるいはさらなる確認が必要か？
        *   **総合的な医学的判断:** 患者背景、臨床経過全体（安全性・有効性データを含む）を考慮し、**参加者の状態に関する医学的な懸念事項、診断や評価の妥当性に関する疑義はないか？** 個々のデータポイントだけでは見えない、全体像としての問題はないか？

    *   **【データ整合性】 (DM視点)**
        *   **クロスドメイン整合性:** 異なるドメイン間のデータ（日付、ID、関連イベント等）に形式的・論理的な矛盾はないか？ **これらの矛盾が、医学的評価やプロトコル遵守の判断に影響を与えないか？** (例: AE発生日 vs LB/VS測定日、AE回復日 vs LB/VS測定日、AE vs CM開始/終了日、MH vs AE/CM、DM.SEX vs 性別依存のイベント/検査、AE発現日 vs EX投与期間)
        *   **ドメイン内整合性:** 各ドメイン内のデータに形式的・論理的な矛盾はないか？ (例: AE 開始日 <= AE 終了日、投与量と単位の一貫性)
        *   **異常値/外れ値:** 定義された範囲外の値、統計的に極端な値、または現実的にありえない値はないか？ **これらがデータ入力エラーなのか、それとも医学的に重要な情報（【医学的レビュー】で評価）なのかを区別する必要があるか？**
        *   **欠損値:** プロトコル上必須、または医学的・統計的評価に重要な変数に欠損はないか？ **欠損が参加者の安全性評価や試験の有効性評価の信頼性に影響を与えないか？** 欠損が許容されるか、理由が適切か？
        *   **評価基準適用のためのデータ:** 有効性・安全性評価の基準（例: RECIST、CTCAE Grade、スコアリング基準）を適用するために必要な元データは揃っており、計算や評価結果と整合しているか？ **データの不備により評価の信頼性が損なわれていないか？**

    *   **【プロトコル遵守】 (CRA視点)**
        *   **選択/除外基準:** 患者はプロトコルで規定された選択基準を満たし、除外基準に該当していないか？ **基準違反が参加者の安全性や試験結果の解釈に与える影響は？** (DM, MHなどを参照)
        *   **治験薬投与:** 投与レジメン（薬剤、用量、経路、頻度、期間）はプロトコルで規定された通りか？ 投与量の変更・中断・再開は適切に記録されているか？ **逸脱がある場合、参加者の安全性や有効性評価への影響は？** (EXを参照)
        *   **併用禁止/制限薬:** プロトコルで禁止または制限されている薬剤が使用されていないか？ **使用されている場合、参加者の安全性リスクや有効性評価への影響は？** (CMを参照)
        *   **評価スケジュール/手順:** 各評価（安全性、有効性、その他）はプロトコルで規定されたVisit、タイミング（Visit Window含む）、および手順（例: 特定の評価機器、評価方法変数 [--METHOD]）で実施されているか？ **逸脱や欠損がある場合、参加者の安全性監視や評価の信頼性にどのような影響があるか？** (VISIT情報、各ドメインの--DY/VISITNUMなどを参照)
        *   **同意/中止等:** 同意取得日と治験手順開始日の関係、中止基準の遵守、中止理由の記録などは適切か？ **参加者の権利が保護されているか？**

**3. 疑義事項の分類とクエリ/内部確認事項の作成:**

*   参照情報: 統合レビューの結果、JSONデータ、Define.xml、プロトコル
*   タスク:
    *   **統合レビューで特定された問題点や疑義事項についてのみ**、以下の分類を行ってください。
        *   **医療機関へのクエリ:** **参加者の安全性確保、評価の信頼性担保、またはデータの正確性・完全性の確認のために、医療機関への問い合わせが必要**な事項。
        *   **内部確認事項:** 医療機関への問い合わせは不要だが、内部で確認・記録すべき事項（例：軽微なデータ不整合で影響が小さい、解釈に関する内部での議論が必要）。

    *   特定した各指摘事項およびそれに基づくクエリ/内部確認事項に対して、**後述の重要度の定義に基づき重要度（Critical/Major/Minor）を付与してください。** 判断は、**参加者の安全性への潜在的な影響度、および試験の評価項目（有効性・安全性）の信頼性への影響度**を総合的に考慮してください。（この重要度は、関連する統合レビューの指摘事項の重要度と一致させるか、クエリ/確認事項としての影響を再評価して決定してください。）
    *   作成された医療機関へのクエリおよび内部確認事項は、それぞれ重要度が高いもの (Critical > Major > Minor) から順にリストアップし、連番 (Q-1, Q-2,... / I-1, I-2,...) を付与してください。
    *   レビューの結果、クエリや内部確認事項を作成する必要がない場合は、「疑義事項なし」と明確に回答してください。

**重要度の定義:**
*   **Critical (致命的/重大):**
    *   **影響:** **参加者の権利、安全性、健康に**重大なリスク**をもたらす、またはその可能性が極めて高い。** または、**データの信頼性や完全性を著しく損ない**、試験結果（特に**主要評価項目**）の解釈に**重大な影響**を与え、**試験の科学的妥当性を脅かす。** 規制当局への報告義務やGCP遵守に**重大な影響**を与える。**(補足参照: Critical to Quality Factor に関連する問題を含む)**
    *   **具体的な状況例:**
        *   重篤な有害事象 (SAE) の未報告、または評価に関する**医学的に重大な疑義や参加者の安全性を脅かす可能性のある不備**。
        *   主要な選択/除外基準の明確な違反で、**参加者への重大なリスクや主要評価項目の信頼性への重大な影響が懸念される**もの。
        *   治験薬の重大な誤投与で、**医学的に重大な結果を招く可能性が高い**もの。
        *   **主要有効性評価項目**データの欠損、重大な不整合、または信頼性への疑義で、**試験結果の解釈を根本的に覆す可能性がある**もの。
        *   同意取得前の治験関連手順の実施など、**参加者の権利を著しく侵害する**もの。
        *   **データから強く示唆される、未報告の重篤な安全性シグナル。**
    *   **対応:** 通常、即時のアクション（例: 緊急クエリ、プロトコル逸脱報告）が必要。

*   **Major (主要):**
    *   **影響:** **参加者の権利、安全性、健康に**潜在的なリスク**をもたらす可能性がある（Criticalほどではない）。** または、**データの信頼性や完全性に影響**を与え、試験結果（特に**副次評価項目**や重要な安全性評価項目）の解釈に**影響を与える可能性**があり、**評価の信頼性を損なう。** プロトコルからの**重要な逸脱**に該当する。
    *   **具体的な状況例:**
        *   非重篤な有害事象の評価と他の臨床データとの明らかな矛盾で、**医学的な判断や安全性評価に影響を与える可能性がある**もの。
        *   **副次有効性評価項目**や**重要な安全性評価項目**に関連するデータの不整合や欠損、プロトコル規定からの逸脱（評価時期、手順など）で、**評価の信頼性に影響を与える**もの。
        *   併用禁止/制限薬の使用（**医学的な安全性リスクが中程度以下と判断されるが、確認が必要**な場合、または**有効性評価への影響が懸念される**場合）。
        *   重要な検査・評価（安全性・有効性）の未実施や、規定されたVisit Windowからの逸脱で、**医学的評価や評価の信頼性に影響を与える可能性がある**もの。
        *   投与量変更や一時中断/再開に関する記録の不備や矛盾で、**参加者の曝露量評価や安全性評価、有効性評価に影響する**もの。
        *   **データパターンから示唆される、潜在的な安全性懸念や有効性に関する疑問点。**
    *   **対応:** 通常、クエリ発行による確認・修正や、内部での詳細な調査が必要。

*   **Minor (軽微):**
    *   **影響:** **参加者の安全性や試験結果の解釈への**直接的な影響は小さい**と考えられる。** 主にデータの**品質や一貫性**に関わる問題で、**評価の信頼性への影響は限定的**。プロトコルからの**軽微な逸脱**で、試験の主要な目的や医学的評価に大きな影響を与えないもの。
    *   **具体的な状況例:**
        *   明らかな誤字脱字（ただし、事象名や薬剤名、有効性評価の重要なキーワードなど、解釈に影響を与えうる場合はMajor以上と判断することもある）。
        *   重要度の低いデータの欠損や軽微な不整合（例: 終了日が開始日より前だが、臨床的な時間経過から明らかに誤記と判断でき、医学的影響が小さい）。
        *   臨床的に意義の小さい検査値/バイタルサインの記録に関する軽微な矛盾で、**医学的な判断や評価の信頼性に影響しない**もの。
        *   選択/除外基準を判定するための検査結果が報告されておらず、適格性判定が不能な場合。（ただし、他のデータから選択/除外基準違反が強く示唆される場合はMajor以上と判断することもある）
        *   Visit日付のわずかなずれ（プロトコルで許容範囲が定義されていない、または有効性評価の厳密性が低い場合など、**医学的評価や評価の信頼性への影響が無視できる**場合）。
    *   **対応:** 内部確認事項として記録するか、他のクエリと併せて確認する、または修正不要と判断する場合もある。

**補足: Critical to Quality Factor (CtQF) について**
*   CtQF は、試験の品質に不可欠な要素であり、これらに関連する重大な問題は原則として Critical と判断されます。CtQF の例としては以下のようなものが挙げられますが、これらは主に試験全体の品質管理に関わる要素です。個別の症例レビューにおいては、これらの要素が患者の安全性、権利、データの信頼性、特に主要評価項目に与える**具体的な影響度**を考慮して重要度を判断してください。
*   **(CtQF の例 - 主要なもの)**
    *   評価の質（標準化、一貫性）に関する重大な問題
    *   組み入れ/除外基準の厳格な適用に関する重大な違反
    *   併用禁止薬の使用
    *   中止基準違反
    *   主要評価項目に影響を与える重大な欠測やデータ品質の問題
    *   同意プロセスに関する重大な不備


**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

# [USUBJID]のデータ統合レビュー報告

## 1. 症例サマリー

*   **患者背景:**
主要な背景情報を**Define.xmlのラベル名を用いて**簡潔に記載する。
（例: 68歳、女性、白人（NOT HISPANIC OR LATINO）。治験実施国はUSAであり、実際に割り付けられた治療群はHigh Doseであった。主要な既往歴として、心筋梗塞（1986年発症）、冠動脈バイパス術（2006年実施）が報告されている。）

*   **イベント推移:**

|日付（YYYY年MM月DD日）|Study Day (Visit名)|イベント内容|
|:---|:---|:---|
|YYYY年MM月DD日|Day XX (N/A)|イベント内容 (例: 有害事象「頭痛」(Severe) 発現)|
|YYYY年MM月DD日|Day YY (UNSCHEDULED 1.1)|イベント内容 (例: ALT値上昇 (Grade 1, 基準値上限の1.5倍))|
|YYYY年MM月DD日|Day ZZ (WEEK 2)|イベント内容 (例: RECIST評価: Stable Disease (SD))|
|... (時系列で記載)|...|...|

## 2. 統合レビュー結果

*   **【医学的レビュー】からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** M-1
            *   **重要度:** [Critical/Major/Minor]
            *   **内容:** [具体的な医学的懸念事項（安全性・有効性含む）、評価の妥当性に関する指摘、潜在的リスクの指摘など。参加者保護や評価信頼性の観点から。]
            *   **根拠:** [判断の根拠、一般的な医学的知見などを記載。]
            *   **関連データ:**
                * [ラベル名(変数名)] = 値（例：[有害事象名(AETERM)] = '頭痛'）
                *   ... (複数の項目が関連する場合は箇条書きで追加する)
        *   ... (複数の指摘事項があれば M-2, M-3...)

*   **【データ整合性】観点からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** D-1
            *   **重要度:** [Critical/Major/Minor]
            *   **内容:** [具体的なデータの不整合、異常値、欠損値、フォーマットの問題などに関する指摘。その問題が医学的評価や評価の信頼性にどのような影響を持ちうるかも簡潔に触れる。]
            *   **根拠:** [判断の根拠、一般的な医学的知見などを記載。]
            *   **関連データ:**
                * [ラベル名(変数名)] = 値（例：[有害事象開始日(AESTDTC)] = '2023-10-26'）
                *   ... (複数の項目が関連する場合は箇条書きで追加する)
        *   ... (複数の指摘事項があれば D-2, D-3...)

*   **【プロトコル遵守】観点からの指摘事項 (逸脱の可能性):**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** P-1
            *   **重要度:** [Critical/Major/Minor]
            *   **逸脱の可能性:** [具体的なプロトコルからの逸脱の可能性。その逸脱が参加者の安全性や評価の信頼性にどのような影響を持ちうるかも簡潔に触れる。]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **根拠:** [判断の根拠、一般的な医学的知見などを記載。]
            *   **関連データ:**
                * [ラベル名(変数名)] = 値（例：[投与開始日(CM.CMSTDTC)] = '2023-11-01'）
                *   ... (複数の項目が関連する場合は箇条書きで追加する)
        *   ... (複数の指摘事項があれば P-2, P-3...)

## 3. 疑義事項

*   [クエリも内部確認事項もない場合は「疑義事項なし」と記載]
*   **医療機関へのクエリ:**
    *   [クエリがない場合は「クエリなし」と記載]
    *   (クエリがある場合)
        *   **クエリNo.:** Q-1 (関連指摘No.: [例: M-1, D-2, P-1])
            *   **重要度:** [Critical/Major/Minor]
            *   **発行担当者:** [指摘内容に最も関連性の高い役割を記載: Medical Monitor, CRA, DM]
            *   **医療機関への問い合わせ文面:** [具体的かつ客観的な問い合わせ内容。なぜこの情報が必要なのか、どのような**参加者保護**または**評価信頼性**に関する懸念に基づいているのかを明確に含める。**文中で特定のデータに言及する場合は、Define.xmlで定義された変数に対応する「ラベル名」を使用し、「「ラベル名」が「値」ですが、...」といった形式でラベル名を記述（変数名は不要）してください。例：「有害事象名「頭痛」について、重症度が「高度」と記録されていますが、詳細をお知らせください。」「Study Day 50のアラニンアミノトランスフェラーゼが 「150 IU/L」 と高値ですが、臨床的な意義について評価をお願いします。」**]
            *   **クエリ文面（英語）:** [具体的かつ客観的な医療機関への問い合わせ内容（英語）。なぜこの情報が必要なのか、どのような**参加者保護**または**評価信頼性**に関する懸念に基づいているのかを明確に含めてください。**文中で特定のデータに言及する場合は、可能であればDefine.xmlで定義された変数に対応する英語の「ラベル名」（または平易な英語表現）を使用し、「... the (Label Name / description) was (Value)...」といった形式で簡潔（最大でも300文字）に記述してください。例: "Regarding the AE Term 'Headache', the Severity is recorded as 'Severe'. Please provide further details." / "On Study Day 50, the ALT value was 150 IU/L. Please assess the clinical significance."**]
            *   **判断理由:** [なぜ問い合わせが必要かの簡潔な理由。**参加者の安全性確保、評価の信頼性担保、データの正確性確認**の観点から記載。]
            *   **判断根拠:**
                *   関連するデータ: 例: [検査項目(LB.LBTESTCD)] = 'ALT', [検査日(Study Day)(LB.LBDY)] = 50, [検査結果(数値)(LB.LBSTRESN)] = 150, [毒性グレード(LB.LBTOXGR)] = '2'
                *   関連するプロトコル箇所: 例: Protocol Section Y.Z (安全性モニタリング)
                *   関連する医学的知見: 例: 薬剤性肝障害の可能性
        *   ... (複数のクエリがあれば Q-2, Q-3...)

*   **内部確認事項 (問い合わせ不要):**
    *   [内部確認事項がない場合は「内部確認事項なし」と記載]
    *   (内部確認事項がある場合)
        *   **確認事項No.:** I-1 (関連指摘No.: [例: D-1, P-2])
            *   **重要度:** [Critical/Major/Minor]
            *   **確認担当者:** [指摘内容に最も関連性の高い役割を記載: Medical Monitor, CRA, DM]
            *   **疑義事項/確認内容:** [問い合わせは不要だが、記録・確認すべき内容。**参加者保護や評価信頼性への影響が小さいと判断した理由**も簡潔に記載。]
            *   **判断理由:** [なぜ問い合わせ不要か、なぜ記録が必要かの簡潔な理由。]
            *   **判断根拠:**
                *   関連するデータ: 例: [生年月日(DM.BRTHDTC)] = '1958-05-10', [年齢(DM.AGE)] = 65
                *   関連するプロトコル箇所: 例: Section 4.1 選択基準 (年齢 18-75歳)
        *   ... (複数の内部確認事項があれば I-2, I-3...)
'''



## Define.xmlとプロトコルの埋め込み

In [26]:
# --- Define.xmlの再確認 ---
# セル9で読み込んだ define_xml_content を使用
if 'define_xml_content' not in locals() or not define_xml_content:
    print("警告: define.xmlの内容が読み込めていません。プロンプト生成に影響があります。")
    # 必要であればここで再度読み込みを試みるか、エラー処理を行う
    define_xml_content = "<Define><Error>Content not loaded</Error></Define>" # ダミーの内容

UserInput_single_end = '\n---\n\n**臨床試験実施計画書（プロトコル）:**\n\n```\n' + protocol_text +  '\n```\n\n**データ定義ファイル（Define.xml）:**\n\n```xml\n' + define_xml_content + '\n```\n\n**臨床試験データ（JSON形式、SDTM準拠）:**\n\n```json\n'

print("プロンプトテンプレートを定義しました。")
# print("\n--- システムプロンプト (最初の100文字) ---")
# print(SysPrompt_single[:100] + "...")
# print("\n--- ユーザー入力テンプレート (最初の100文字) ---")
# print(UserInput_single[:100] + "...")

プロンプトテンプレートを定義しました。


# Geminiの実行

In [27]:
# --- Geminiモデル設定 ---
#MODEL_NAME = 'gemini-2.5-pro-exp-03-25'
MODEL_NAME = 'gemini-2.0-flash'

# --- 生成パラメータ設定 ---
GENERATION_TEMPERATURE = 0.2  # 創造性の調整 (低いほど決定的)
GENERATION_TOP_P = 0.9        # トークン選択の確率調整 (高いほど多様)
# GENERATION_MAX_OUTPUT_TOKENS = 8192 # 必要に応じて最大出力トークン数を設定

generation_config = genai.types.GenerationConfig(
    temperature=GENERATION_TEMPERATURE,
    top_p=GENERATION_TOP_P,
    # max_output_tokens=GENERATION_MAX_OUTPUT_TOKENS # 必要に応じて設定
)

# --- 処理対象被験者の特定 ---
# Target_data から更新があった被験者リストを作成
if 'Target_data' in locals() and Target_data:
    updated_subjects_set = {item[1] for item in Target_data if len(item) > 1}
    updated_subjects = sorted(list(updated_subjects_set))
    print(f"データ更新があった被験者リスト (Target_dataより): {updated_subjects}")
else:
    print("警告: Target_data が未定義または空です。処理対象の被験者リストが空になります。")
    updated_subjects = []

# --- 実行対象の設定 (テスト用) ---
# updated_subjects = ['01-704-1017','01-703-1042','01-701-1111'] # 特定の被験者のみ実行する場合
# updated_subjects = updated_subjects[:1] # 最初のN件のみ実行する場合 (例: 最初の1件)

print(f"\nGeminiモデル: {MODEL_NAME}")
print(f"生成設定: Temperature={GENERATION_TEMPERATURE}, Top_P={GENERATION_TOP_P}")
# if 'GENERATION_MAX_OUTPUT_TOKENS' in locals():
#      print(f"最大出力トークン数: {GENERATION_MAX_OUTPUT_TOKENS}")
print(f"レビュー対象被験者 ({len(updated_subjects)}名): {updated_subjects}")

データ更新があった被験者リスト (Target_dataより): ['01-701-1015', '01-701-1023', '01-701-1028', '01-701-1034', '01-701-1047', '01-701-1097', '01-701-1111', '01-701-1118', '01-701-1146', '01-701-1148', '01-701-1153', '01-701-1180', '01-701-1181', '01-701-1363', '01-701-1383', '01-701-1387', '01-702-1082', '01-703-1042', '01-703-1076', '01-703-1086', '01-703-1096', '01-703-1258', '01-703-1279', '01-703-1299', '01-703-1335', '01-703-1403', '01-704-1008', '01-704-1009', '01-704-1010', '01-704-1017']

Geminiモデル: gemini-2.0-flash
生成設定: Temperature=0.2, Top_P=0.9
レビュー対象被験者 (30名): ['01-701-1015', '01-701-1023', '01-701-1028', '01-701-1034', '01-701-1047', '01-701-1097', '01-701-1111', '01-701-1118', '01-701-1146', '01-701-1148', '01-701-1153', '01-701-1180', '01-701-1181', '01-701-1363', '01-701-1383', '01-701-1387', '01-702-1082', '01-703-1042', '01-703-1076', '01-703-1086', '01-703-1096', '01-703-1258', '01-703-1279', '01-703-1299', '01-703-1335', '01-703-1403', '01-704-1008', '01-704-1009', '01-704-1010', '

## Geminiに送る

In [29]:
import google.generativeai as genai
import time
import json # json.dumps を使うためにインポート

# --- リトライ設定 ---
MAX_RETRIES = 3
RETRY_DELAY = 10 # 秒

print("\n--- Geminiレビュー実行開始 ---")

results_list = []
processed_count = 0
error_count = 0
failed_subjects = [] # リトライしても失敗した症例IDを格納するリスト

# dataset_list_updated と protocol_text が存在するか確認
if 'dataset_list_updated' not in locals() or not dataset_list_updated:
    print("エラー: 更新後のデータリスト (dataset_list_updated) が存在しません。レビューを実行できません。")
elif 'protocol_text' not in locals():
     print("エラー: プロトコルテキスト (protocol_text) が存在しません。レビューを実行できません。")
elif not updated_subjects:
    print("情報: レビュー対象の被験者がいません。処理をスキップします。")
else:
    # GenerativeModelインスタンスを作成 (システムプロンプトとモデル名を指定)
    try:
        # modelインスタンスの作成はループの外で行う（毎回作成する必要はない）
        model = genai.GenerativeModel(
            model_name=MODEL_NAME,
            system_instruction=SysPrompt_single,
            generation_config=generation_config # 生成設定を渡す
        )
        print(f"モデル '{MODEL_NAME}' の初期化完了。")
    except Exception as e:
        print(f"致命的エラー: モデル '{MODEL_NAME}' の初期化に失敗しました: {e}")
        # モデル初期化失敗時はループを実行しない
        updated_subjects = [] # ループが実行されないように空にする

    # 各被験者についてレビューを実行
    total_subjects = len(updated_subjects)
    for i, subj_id in enumerate(updated_subjects):
        print(f"\n[{i+1}/{total_subjects}] 被験者 '{subj_id}' のレビューを開始...")
        start_time = time.time() # 各被験者の処理開始時間

        # 1. 対象被験者のデータを抽出
        subject_data_list = filter_data(dataset_list_updated, [subj_id])
        if not subject_data_list:
            print(f"  警告: 被験者 '{subj_id}' のデータ抽出に失敗しました。スキップします。")
            # エラーとして記録し、失敗リストに追加
            results_list.append(f"# [{subj_id}] のデータ統合レビュー報告\n\nエラー: 対象データの抽出に失敗しました。")
            error_count += 1
            failed_subjects.append(subj_id + " (データ抽出失敗)")
            continue

        # 2. プロンプトを組み立てる
        try:
            subject_data_json = json.dumps(subject_data_list, ensure_ascii=False, indent=2)
            prompt_parts = [
                UserInput_single,
                UserInput_single_end,
                subject_data_json,
                 '\n```'
            ]
        except Exception as e:
             print(f"  エラー: 被験者 '{subj_id}' のプロンプト組み立て中にエラーが発生しました: {e}")
             # エラーとして記録し、失敗リストに追加
             results_list.append(f"# [{subj_id}] のデータ統合レビュー報告\n\nエラー: プロンプト組み立て失敗: {e}")
             error_count += 1
             failed_subjects.append(subj_id + " (プロンプト組み立て失敗)")
             continue

        # 3. Gemini API呼び出し (リトライロジック付き)
        response = None # response変数を初期化
        last_exception = None # 最後のエラーを保持するため
        for attempt in range(MAX_RETRIES + 1): # 初回実行 + MAX_RETRIES回のリトライ
            try:
                if attempt > 0:
                     print(f"    リトライ {attempt}/{MAX_RETRIES} 回目...")

                api_start_time = time.time()
                response = model.generate_content(prompt_parts) # 部品リストで渡す
                api_end_time = time.time()
                api_elapsed_time = api_end_time - api_start_time

                # ---- 成功時の処理 ----
                # 4. 結果の評価と格納
                if response.text:
                    results_list.append(response.text)
                    total_elapsed_time = time.time() - start_time
                    print(f"  レビュー完了 ({total_elapsed_time:.2f} 秒, API:{api_elapsed_time:.2f} 秒)")
                    processed_count += 1
                    # トークン数の表示（利用可能な場合）
                    if hasattr(response, 'usage_metadata') and response.usage_metadata:
                        print(f"    Tokens: Prompt={response.usage_metadata.prompt_token_count}, Candidates={response.usage_metadata.candidates_token_count}, Total={response.usage_metadata.total_token_count}")
                    else:
                        print("    トークン使用量メタデータは利用できません。")
                    break # ★★★ 成功したらリトライループを抜ける ★★★
                else:
                     # レスポンスはあるがテキストがない場合（ブロックされた等）
                     # これはAPIエラーではないためリトライ対象外とし、警告として記録
                    total_elapsed_time = time.time() - start_time
                    print(f"  警告: 被験者 '{subj_id}' のレビュー結果が空です。({total_elapsed_time:.2f} 秒, API:{api_elapsed_time:.2f} 秒)")
                    # 詳細情報を取得
                    finish_reason = "UNKNOWN"
                    block_reason = "NONE"
                    if response.candidates:
                         candidate = response.candidates[0]
                         finish_reason = candidate.finish_reason.name
                         if hasattr(candidate, 'safety_ratings'):
                             safety_ratings = candidate.safety_ratings
                             block_reason = next((r.category.name for r in safety_ratings if r.probability.name != "NEGLIGIBLE"), "NONE")

                    results_list.append(f"# [{subj_id}] のデータ統合レビュー報告\n\n警告: レビュー結果が空でした。\nFinish Reason: {finish_reason}\nBlock Reason: {block_reason}")
                    error_count += 1
                    failed_subjects.append(subj_id + f" (結果空: {finish_reason}/{block_reason})")
                    break # ★★★ このケースもリトライせずループを抜ける ★★★

            except Exception as e:
                # ---- 例外発生時の処理 ----
                last_exception = e # 最後のエラーを更新
                api_end_time = time.time()
                api_elapsed_time = api_end_time - api_start_time
                print(f"    エラー発生 (試行 {attempt+1}/{MAX_RETRIES+1}, API:{api_elapsed_time:.2f} 秒): {e}")

                if attempt < MAX_RETRIES:
                    # まだリトライ可能な場合
                    print(f"    {RETRY_DELAY}秒後にリトライします...")
                    time.sleep(RETRY_DELAY)
                    # ループの次の繰り返しへ
                else:
                    # 最大リトライ回数に達した場合
                    print(f"  エラー: 被験者 '{subj_id}' のレビュー中にAPIエラーが{MAX_RETRIES+1}回発生しました。最終エラー: {last_exception}")
                    total_elapsed_time = time.time() - start_time
                    # エラーとして記録し、失敗リストに追加
                    results_list.append(f"# [{subj_id}] のデータ統合レビュー報告\n\nエラー: API呼び出し失敗 ({MAX_RETRIES+1}回試行後): {last_exception}")
                    error_count += 1
                    failed_subjects.append(subj_id + f" (APIエラー: {last_exception})")
                    # リトライループはここで終了（breakは不要、ループ条件で抜ける）

# --- ループ完了後 ---
print(f"\n--- Geminiレビュー実行終了 ---")
print(f"処理完了: {processed_count} 件成功, {error_count} 件エラー/警告")

# 実行できなかった（失敗した）症例リストを表示
if failed_subjects:
    print("\n--- 処理に失敗した症例リスト ---")
    # 重複を除いて表示（データ抽出失敗とAPIエラー両方発生する可能性も考慮）
    unique_failed_subjects = sorted(list(set(failed_subjects)))
    for failed_info in unique_failed_subjects:
        print(f"- {failed_info}")
else:
    print("\nすべての症例の処理が完了しました（失敗なし）。")


--- Geminiレビュー実行開始 ---
モデル 'gemini-2.0-flash' の初期化完了。

[1/30] 被験者 '01-701-1015' のレビューを開始...
  レビュー完了 (18.19 秒, API:13.78 秒)
    Tokens: Prompt=290234, Candidates=1720, Total=291954

[2/30] 被験者 '01-701-1023' のレビューを開始...
  レビュー完了 (15.80 秒, API:11.20 秒)
    Tokens: Prompt=180129, Candidates=2345, Total=182474

[3/30] 被験者 '01-701-1028' のレビューを開始...
  レビュー完了 (14.32 秒, API:10.00 秒)
    Tokens: Prompt=296246, Candidates=1658, Total=297904

[4/30] 被験者 '01-701-1034' のレビューを開始...
  レビュー完了 (37.93 秒, API:33.82 秒)
    Tokens: Prompt=306538, Candidates=7977, Total=314515

[5/30] 被験者 '01-701-1047' のレビューを開始...
  レビュー完了 (22.75 秒, API:19.22 秒)
    Tokens: Prompt=189302, Candidates=4639, Total=193941

[6/30] 被験者 '01-701-1097' のレビューを開始...


    エラー発生 (試行 1/4, API:3.41 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (43.04 秒, API:24.74 秒)
    Tokens: Prompt=296853, Candidates=4973, Total=301826

[7/30] 被験者 '01-701-1111' のレビューを開始...
  レビュー完了 (21.51 秒, API:18.26 秒)
    Tokens: Prompt=168599, Candidates=4407, Total=173006

[8/30] 被験者 '01-701-1118' のレビューを開始...
  レビュー完了 (14.30 秒, API:9.57 秒)
    Tokens: Prompt=291854, Candidates=774, Total=292628

[9/30] 被験者 '01-701-1146' のレビューを開始...
  レビュー完了 (20.55 秒, API:15.79 秒)
    Tokens: Prompt=186393, Candidates=3404, Total=189797

[10/30] 被験者 '01-701-1148' のレビューを開始...


    エラー発生 (試行 1/4, API:2.46 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (48.33 秒, API:32.62 秒)
    Tokens: Prompt=308491, Candidates=8068, Total=316559

[11/30] 被験者 '01-701-1153' のレビューを開始...


    エラー発生 (試行 1/4, API:2.36 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (31.92 秒, API:16.27 秒)
    Tokens: Prompt=306890, Candidates=3417, Total=310307

[12/30] 被験者 '01-701-1180' のレビューを開始...
  レビュー完了 (19.25 秒, API:14.42 秒)
    Tokens: Prompt=175323, Candidates=3449, Total=178772

[13/30] 被験者 '01-701-1181' のレビューを開始...
  レビュー完了 (17.20 秒, API:13.41 秒)
    Tokens: Prompt=166541, Candidates=3415, Total=169956

[14/30] 被験者 '01-701-1363' のレビューを開始...


    エラー発生 (試行 1/4, API:1.82 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (30.94 秒, API:15.34 秒)
    Tokens: Prompt=308617, Candidates=3196, Total=311813

[15/30] 被験者 '01-701-1383' のレビューを開始...


...

    エラー発生 (試行 2/4, API:2.75 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 2/3 回目...
  レビュー完了 (46.45 秒, API:18.81 秒)
    Tokens: Prompt=295573, Candidates=3452, Total=299025

[16/30] 被験者 '01-701-1387' のレビューを開始...


    エラー発生 (試行 1/4, API:1.51 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (35.54 秒, API:19.21 秒)
    Tokens: Prompt=158158, Candidates=4960, Total=163118

[17/30] 被験者 '01-702-1082' のレビューを開始...


    エラー発生 (試行 1/4, API:1.43 秒): 499 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: The operation was cancelled.
    10秒後にリトライします...
    リトライ 1/3 回目...
  レビュー完了 (40.17 秒, API:24.98 秒)
    Tokens: Prompt=227824, Candidates=6244, Total=234068

[18/30] 被験者 '01-703-1042' のレビューを開始...
  レビュー完了 (18.52 秒, API:14.60 秒)
    Tokens: Prompt=302291, Candidates=2019, Total=304310

[19/30] 被験者 '01-703-1076' のレビューを開始...
  レビュー完了 (20.20 秒, API:16.97 秒)
    Tokens: Prompt=217313, Candidates=3473, Total=220786

[20/30] 被験者 '01-703-1086' のレビューを開始...
  レビュー完了 (19.01 秒, API:14.23 秒)
    Tokens: Prompt=253217, Candidates=2805, Total=256022

[21/30] 被験者 '01-703-1096' のレビューを開始...
  レビュー完了 (20.57 秒, API:17.24 秒)
    Tokens: Prompt=158363, Candidates=4255, Total=162618

[22/30] 被験者 '01-703-1258' のレビューを開始...
  レビュー完了 (24.17 秒, API:19.59 秒)
    Tokens: Prompt=318542, Candidates=3290, Total=321832

[23/30] 被験者 '01-703-1279' のレビューを開始...

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 1859.78ms


  レビュー完了 (18.45 秒, API:15.20 秒)
    Tokens: Prompt=177962, Candidates=3069, Total=181031

[26/30] 被験者 '01-703-1403' のレビューを開始...
  レビュー完了 (15.33 秒, API:11.04 秒)
    Tokens: Prompt=126738, Candidates=2763, Total=129501

[27/30] 被験者 '01-704-1008' のレビューを開始...
  レビュー完了 (21.83 秒, API:18.53 秒)
    Tokens: Prompt=180439, Candidates=4769, Total=185208

[28/30] 被験者 '01-704-1009' のレビューを開始...
  レビュー完了 (28.07 秒, API:23.33 秒)
    Tokens: Prompt=148004, Candidates=7004, Total=155008

[29/30] 被験者 '01-704-1010' のレビューを開始...
  レビュー完了 (32.39 秒, API:27.97 秒)
    Tokens: Prompt=276687, Candidates=8143, Total=284830

[30/30] 被験者 '01-704-1017' のレビューを開始...
  レビュー完了 (17.10 秒, API:13.73 秒)
    Tokens: Prompt=172869, Candidates=3237, Total=176106

--- Geminiレビュー実行終了 ---
処理完了: 30 件成功, 0 件エラー/警告

すべての症例の処理が完了しました（失敗なし）。


In [30]:
# --- 結果の結合 ---
# results_list に各被験者のレビュー結果(Markdown形式)が格納されている想定
if 'results_list' in locals() and results_list:
    # 各結果の間に空行を2行入れて結合
    combined_markdown = "\n\n---\n\n".join(results_list)

    # --- ファイルへの保存 ---
    # MODEL_NAME が定義されているか確認
    model_name_suffix = MODEL_NAME.replace("/", "_") if 'MODEL_NAME' in locals() else "unknown_model"
    output_filename = f'output_review_{model_name_suffix}.md'

    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(combined_markdown)
        print(f"\nレビュー結果をMarkdownファイルに保存しました: {output_filename}")
    except Exception as e:
        print(f"エラー: レビュー結果のファイル保存に失敗しました: {e}")

    # --- 結果の表示（任意、長くなる可能性あり） ---
    print(f"\n--- 結合されたレビュー結果 (最初の1000文字) ---")
    print(combined_markdown[:1000] + "...")
    # print(combined_markdown) # 全文表示したい場合

else:
    print("レビュー結果リストが存在しないか空のため、結合・保存・表示は行われませんでした。")


レビュー結果をMarkdownファイルに保存しました: output_review_gemini-2.0-flash.md

--- 結合されたレビュー結果 (最初の1000文字) ---
```markdown
# 01-701-1015のデータ統合レビュー報告

## 1. 症例サマリー

*   **患者背景:**
63歳、女性、白人（HISPANIC OR LATINO）。治験実施国はUSAであり、実際に割り付けられた治療群はPlaceboであった。主要な既往歴として、PALPITATIONS、SUBTOTAL HYSTERECTOMY、HEADACHE、TINNITUS、HEARTBURN、THYROIDECTOMY PARTIAL、SORE THROAT、TONSILLECTOMY、NUMBNESS IN LEG、GALLBLADDER STONES、ALZHEIMER'S DISEASEが報告されている。教育レベルは16 YEARS。

*   **イベント推移:**

|日付（YYYY年MM月DD日）|Study Day (Visit名)|イベント内容|
|:---|:---|:---|
|2014年01月03日|Day 2 (BASELINE)|有害事象「APPLICATION SITE ERYTHEMA」(MILD) 発現|
|2014年01月03日|Day 2 (BASELINE)|有害事象「APPLICATION SITE PRURITUS」(MILD) 発現|
|2014年01月09日|Day 8 (BASELINE)|有害事象「DIARRHOEA」(MILD) 発現|
|2014年01月11日|Day 10 (BASELINE)|有害事象「DIARRHOEA」(MILD) 軽快|
|2014年01月16日|Day 15 (WEEK 2)|アラニンアミノトランスフェラーゼが 「41 U/L」 と基準範囲超 (基準範囲上限: 34 U/L)|
|2014年03月05日|Day 63 (WEEK 8)|バイタルサイン：体温が「36.67 C」|
|2014年05月07日|Day 126 (WEEK 16)|バイタルサイン：体温が「36.61 C」|
|2014年05月07日|Day 126 (WEEK 16)|バイタルサイン：臥位での収縮期血

In [ ]:
stopflg

In [ ]:
import google.generativeai as genai
import time
import sys

genai.configure(api_key=api_key)

# --- TemperatureとTop_Pの設定 ---
generation_temperature = 0.3  # 例: 創造性を調整 (0.0-1.0)
generation_top_p = 0.8        # 例: トークン選択の確率を調整 (0.0-1.0)

#updated_subjects = ['01-704-1017','01-703-1042','01-701-1111',]
updated_subjects = ['01-701-1111',]
results_list = []
for subj in updated_subjects:
  datasetjson = filter_data(dataset_list_updated, subj)

  # GenerativeModelインスタンスを作成する際にsystem_instructionを指定
  model = genai.GenerativeModel(
      model_name=model_name,
#-      system_instruction=SysPrompt_single
  )

  # --- GenerationConfigの作成 ---
  # ここでTemperatureとTop_Pを指定します
  generation_config = genai.types.GenerationConfig(
      temperature=generation_temperature,
      top_p=generation_top_p
      # 必要であれば他のパラメータも指定できます (例: max_output_tokens, stop_sequences)
      # max_output_tokens=1024,
      # stop_sequences=["```"]
  )

  UserInput_single = "この患者は基礎疾患に聴覚不全があり、COMMANDSおよびCOMPREHENSION OF SPOKEN LANGUAGEのスコアが低いです。これはデータとして不自然な状況でしょうか？あなたの持つ一般的な医学知識を最大限活用し、聴覚不全やスコアリングのデータをもとに説明してください。"
  UserInput = UserInput_single + UserInput_single_end1 + json.dumps(datasetjson) + UserInput_single_end2 + '*   臨床試験実施計画書（プロトコル）:\n\n```\n' + protocol + '\n```'
  print(f"\nモデル '{model_name}' が '{subj}' のデータをレビューしています...")
  print("-" * 30)
  start_time = time.time()
  # print(f"【システムプロンプト】\n{SysPrompt_single}")
  # print("-" * 30)
  # print(f"【ユーザープロンプト】\n{UserInput}")
  # print("-" * 30)

  # コンテンツ生成を実行し、generation_configを渡す
  try:
      response = model.generate_content(
          UserInput,
          generation_config=generation_config # ★変更点: generation_configを渡す
      )
      end_time = time.time()
      elapsed_time = end_time - start_time

      # --- 6. 結果の表示 ---
      print("結果が生成されました")
      print(f"所要時間: {elapsed_time:.4f} 秒")
      # response.usage_metadata が存在するか確認
      if hasattr(response, 'usage_metadata') and response.usage_metadata:
          print(f"Prompt Token Count: {response.usage_metadata.prompt_token_count}")
          # print(f"Candidates Token Count: {response.usage_metadata.candidates_token_count}")
          print(f"Total Token Count: {response.usage_metadata.total_token_count}")
      else:
          print("Token usage metadata not available.")
      print("-" * 30)
      # response.text で生成されたテキストを取得
      # print(response.text)

      results_list.append(response.text)

  except Exception as e:
      print(f"エラーが発生しました ({subj}): {e}")
      # エラーが発生した場合でもリストにプレースホルダーを追加するなど、必要に応じて処理
      results_list.append(f"Error generating content for {subj}: {e}")


# ループ完了後
print("\nすべての処理が完了しました。")
# print(results_list) # 必要であれば結果リスト全体を出力

print(results_list[0])

In [ ]:
import json

usubjids = set()
for dataset in dataset_list:
    if 'columns' in dataset:
        for i, col in enumerate(dataset['columns']):
            if 'name' in col and col['name'] == 'USUBJID':
                if 'rows' in dataset:
                    for row in dataset['rows']:
                        if len(row) > i:
                            usubjids.add(row[i])

print(list(usubjids))


# 以下メモ

In [ ]:
UserInput = UserInput_single + UserInput_single_end1 + json.dumps(filter_data(dataset_list_updated, ['01-701-1111'])) + UserInput_single_end2 + '*   臨床試験実施計画書（プロトコル）:\n\n```\n' + protocol + '\n```'

In [ ]:
#print(UserInput)

In [ ]:
# 必要なライブラリをインストール
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata # Colabのシークレット機能を使うためにインポート
import os

# ColabのシークレットからAPIキーを読み込む
# userdata.get('SECRET_NAME') の SECRET_NAME は、Colabのシークレットで設定した「名前」に置き換えてください
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError("APIキーがColabシークレットに見つかりません。左側の鍵アイコンから'GOOGLE_API_KEY'という名前で設定してください。")
    genai.configure(api_key=api_key)
    print("APIキーの設定が完了しました。")
except Exception as e:
    print(f"エラー: {e}")
    print("---")
    print("Colabのシークレット機能を使ってAPIキーを設定する方法:")
    print("1. 左側のサイドバーの鍵アイコンをクリックします。")
    print("2. 「新しいシークレットを追加」をクリックします。")
    print("3. 名前: GOOGLE_API_KEY")
    print("4. 値: あなたのAPIキーを貼り付けます。")
    print("5. 「ノートブックへのアクセスを有効にする」をオンにします。")
    print("6. このセルを再度実行してください。")
    # エラーが発生した場合、以降のコード実行を停止させるために例外を再送出してもよい
    # raise e

# --- (代替: 非推奨 - テスト目的でのみ使用し、共有しないでください) ---
# シークレット機能を使わない場合（セキュリティリスクあり）
# api_key = "YOUR_API_KEY" # ここに直接APIキーを貼り付ける（非推奨）
# genai.configure(api_key=api_key)
# print("APIキーの設定が完了しました。")
# --------------------------------------------------------------------
# 使用するモデルを選択 (例: gemini-pro)
# 利用可能なモデルはドキュメントを参照してください: https://ai.google.dev/models/gemini
model = genai.GenerativeModel('gemini-2.0-flash')

# Geminiに送信したいプロンプト（質問や指示）
prompt = "日本の首都はどこですか？簡潔に答えてください。"
# prompt = "Pythonで2つの数値を足し算する簡単な関数を書いてください。"
# prompt = "面白い猫のジョークを教えて。"

print(f"\nプロンプト: {prompt}\n")

try:
    # プロンプトを送信して応答を生成
    response = model.generate_content(prompt)

    # 応答テキストを表示
    print("Geminiからの応答:")
    print(response.text)

    # (オプション) 応答の他の情報（安全性評価など）も確認できます
    # print("\n詳細な応答:")
    # print(response)
    # print("\n候補:")
    # print(response.candidates)
    # print("\n安全性評価:")
    # print(response.prompt_feedback)


except Exception as e:
    print(f"コンテンツ生成中にエラーが発生しました: {e}")
    # エラーの詳細（例: 不適切なコンテンツとしてブロックされた場合など）
    # if hasattr(response, 'prompt_feedback'):
    #     print(f"プロンプトフィードバック: {response.prompt_feedback}")
    # if hasattr(response, 'candidates') and response.candidates:
    #      print(f"フィニッシュリーズン: {response.candidates[0].finish_reason}")
    #      print(f"安全性評価: {response.candidates[0].safety_ratings}")

In [ ]:
#model_name = 'gemini-2.5-pro-exp-03-25'
model_name = 'gemini-2.0-flash'

system_prompt = ''
user_prompt = 'こんにちは'

# GenerativeModelインスタンスを作成する際にsystem_instructionを指定
model = genai.GenerativeModel(
    model_name=model_name,
    system_instruction=system_prompt
)

print(f"\nモデル '{model_name}' に問い合わせています...")
print("-" * 30)
print(f"【システムプロンプト】\n{system_prompt}")
print("-" * 30)
print(f"【ユーザープロンプト】\n{user_prompt}")
print("-" * 30)

# コンテンツ生成を実行
# generate_content にはユーザープロンプトのみを渡します
response = model.generate_content(user_prompt)

# --- 6. 結果の表示 ---
print("【コロからの応答】")
print("-" * 30)
# response.text で生成されたテキストを取得
print(response.text)

In [ ]:
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_single + UserInput_single_end1 + datasetjson + UserInput_single_end2,

In [ ]:
import pandas as pd

results_list = []

updated_subjects = ['01-704-1017','01-703-1042','01-701-1111',]

for subj in updated_subjects:
    datasetjson = filter_data(dataset_list_updated, subj)
    print(f"処理完了：'datasetjson' に USUBJID が {subj} のデータを出力しました。")

    row_data = {'Subject': subj}  # 各行のデータを格納する辞書

    # 統合プロンプトの処理
    workflow_inputs_single = create_workflow_input_single(ModelName, SysPrompt_single, UserInput_single, UserInput_single_end1, json.dumps(datasetjson), UserInput_single_end2)
    try:
        result_Task_single = run_workflow_with_retry(api_key, workflow_inputs_single, user_id)
        output_Task_single = result_Task_single['data']['outputs']['text']
        display(Markdown(output_Task_single))
        row_data['Task_single'] = output_Task_single
    except Exception as e:
        print(f"統合プロンプトの処理でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task_single'] = "Error"
    results_list.append(row_data)

# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

In [ ]:
# --- 1. 必要なライブラリをインストール ---
!pip install -q google-generativeai

# --- 2. ライブラリとモジュールをインポート ---
import google.generativeai as genai
from google.colab import userdata
import sys
import json # JSON文字列のパースに必要
# GenerationConfig と Schema, 安全性設定関連をインポート
from google.generativeai.types import GenerationConfig, Part, HarmCategory, HarmBlockThreshold

# --- 3. ColabのシークレットからAPIキーを読み込む ---
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError("APIキーがシークレットに見つかりません。")
    genai.configure(api_key=api_key)
    print("✅ APIキーの設定が完了しました。")
except ValueError as e:
    print(f"エラー: {e}")
    print("Colabのシークレット（左側の鍵アイコン🔑）に 'GOOGLE_API_KEY' という名前でAPIキーを設定してください。")
    sys.exit(1)
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")
    sys.exit(1)

# --- 4. プロンプトとシステムプロンプトの定義 ---
# JSON抽出タスクに適した指示を与える
system_prompt = """
あなたは優秀な情報抽出AIです。
与えられたテキストから、指定された情報を正確に抽出し、
必ず指定されたJSONスキーマに従って応答してください。
情報が見つからない場合は、該当するフィールドの値を null または空文字にしてください。
"""

# ユーザープロンプト: 抽出対象のテキスト
user_prompt = """
イベントのお知らせ：
来る2024年8月15日、東京ビッグサイトにて「夏の技術フェスタ」を開催します。
参加費は3000円です。基調講演は田中一郎氏が行います。
"""

# --- 5. JSONスキーマの定義 ---
# 出力させたいJSONの構造をSchemaオブジェクトで定義
# (OpenAPI Specificationのスキーマオブジェクトに似た形式)
output_schema = Schema(
    type="OBJECT",
    description="イベント情報", # スキーマ全体の説明 (任意)
    properties={
        'event_name': Schema(type="STRING", description="イベント名"),
        'date': Schema(type="STRING", description="開催日 (YYYY-MM-DD形式)"),
        'location': Schema(type="STRING", description="開催場所"),
        'fee': Schema(type="NUMBER", description="参加費用 (数値)"),
        'keynote_speaker': Schema(type="STRING", description="基調講演者名"),
    },
    required=['event_name', 'date', 'location'] # 必須項目を指定
)

# --- 6. GenerationConfigの設定 ---
generation_config = GenerationConfig(
    temperature=0.1,  # JSON抽出など、正確性が求められる場合は低めに設定
    # top_p=0.9,      # temperatureとtop_pはどちらか一方の指定が推奨されることが多い
    # max_output_tokens=1024, # 必要に応じて最大出力トークン数を制限
    response_mime_type="application/json", # ★ JSON出力を指定
    response_schema=output_schema         # ★ 定義したJSONスキーマを指定
)

# --- 7. モデルの設定とAPI呼び出し ---
try:
    # JSONモードとスキーマ指定をサポートするモデルを選択 (Gemini 1.5 Flash/Proなど)
    model_name = 'gemini-1.5-flash-latest'

    # GenerativeModelインスタンスを作成
    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=system_prompt,
        # JSONモード使用時は、コンテンツフィルターでブロックされる可能性を考慮し、
        # 必要に応じて安全性設定を調整 (BLOCK_NONEはテスト用。本番環境では注意)
        safety_settings={
            HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
            HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
            HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
        }
        # safety_settings を指定しない場合はデフォルト設定が適用されます
    )

    print(f"\nモデル '{model_name}' に問い合わせています...")
    print("-" * 30)
    print(f"【システムプロンプト】\n{system_prompt}")
    print("-" * 30)
    print(f"【ユーザープロンプト】\n{user_prompt}")
    print("-" * 30)
    print(f"【Generation Config】")
    print(f"  Temperature: {generation_config.temperature}")
    # print(f"  Top P: {generation_config.top_p}") # top_pを設定した場合
    print(f"  Response MIME Type: {generation_config.response_mime_type}")
    print("【指定JSONスキーマ】")
    # スキーマの内容を簡易的に表示 (複雑な場合は調整が必要)
    print(f"  Type: {output_schema.type}")
    print(f"  Properties: {list(output_schema.properties.keys())}")
    print(f"  Required: {output_schema.required}")
    print("-" * 30)

    # コンテンツ生成を実行 (generation_config を渡す)
    response = model.generate_content(
        user_prompt,
        generation_config=generation_config
    )

    # --- 8. 結果の表示 (JSON処理) ---
    print("【AIからの応答 (JSON形式)】")
    print("-" * 30)

    # response.text にはJSON形式の文字列が含まれる
    raw_json_output = response.text
    print("Raw JSON Output:")
    print(raw_json_output)
    print("-" * 30)

    # JSON文字列をPythonオブジェクト (辞書) にパース
    try:
        parsed_json = json.loads(raw_json_output)
        print("Parsed JSON Object:")
        # Pythonオブジェクトを見やすくインデントして表示 (ensure_ascii=Falseで日本語をそのまま表示)
        print(json.dumps(parsed_json, indent=2, ensure_ascii=False))

        # パースしたデータへのアクセス例
        # print(f"\n抽出されたイベント名: {parsed_json.get('event_name')}")

    except json.JSONDecodeError as json_e:
        print(f"❌ JSONのパースに失敗しました: {json_e}")
        print("モデルが有効なJSONを出力しなかった可能性があります。プロンプト、スキーマ、またはモデルの互換性を確認してください。")
    except AttributeError:
         # response.text が存在しない場合 (例: ブロックされた場合)
         print("❌ 応答テキストが取得できませんでした。")
         # responseオブジェクト全体を出力して詳細を確認
         print("Response object:", response)


# --- 9. エラーハンドリング ---
except genai.types.generation_types.BlockedPromptException as e:
    print("❌ エラー: プロンプトがブロックされました。")
    print("プロンプトの内容がセーフティポリシーに違反している可能性があります。")
    print(f"詳細: {e}")
    # ブロックされた理由などの詳細情報を含むことがある
    # print(response.prompt_feedback)
except genai.types.generation_types.StopCandidateException as e:
    print("❌ エラー: 応答の生成が途中で停止しました。")
    print("コンテンツフィルター (FINISH_REASON_SAFETY) や他の理由 (FINISH_REASON_RECITATIONなど) で応答が完了しなかった可能性があります。")
    print(f"詳細: {e}")
    # 停止理由などの詳細情報を含むことがある
    # print(response.candidates[0].finish_reason)
    # print(response.candidates[0].safety_ratings)
except Exception as e:
    print(f"❌ エラーが発生しました: {e}")
    if "does not support" in str(e) and "application/json" in str(e):
        print(f"選択したモデル '{model_name}' はJSONモードまたは指定されたスキーマをサポートしていない可能性があります。Gemini 1.5 Pro/Flashなど、サポートしているモデルを確認してください。")
    elif "schema is invalid" in str(e) or "could not be parsed" in str(e):
        print("指定されたJSONスキーマの形式が無効か、モデルが解釈できませんでした。スキーマの定義を確認してください。")
    elif "API key not valid" in str(e):
        print("APIキーが無効か、正しく設定されていない可能性があります。Colabシークレットの設定を確認してください。")
    elif "permission denied" in str(e) or "403" in str(e):
         print(f"APIキーに、選択したモデル ('{model_name}') を使用する権限がないか、JSONモードの利用が許可されていない可能性があります。Google Cloud ConsoleやAI Studioの設定を確認してください。")
    elif "Resource has been exhausted" in str(e) or "429" in str(e):
         print("APIの利用制限（例: 1分あたりのリクエスト数）に達した可能性があります。少し時間をおいてから再度試してください。")
    else:
        print("予期せぬAPIエラーが発生しました。")

In [ ]:
# --- 1. 必要なライブラリをインストール ---
!pip install -q google-generativeai

# --- 2. ライブラリとモジュールをインポート ---
import google.generativeai as genai
from google.colab import userdata # Colabのシークレットを読み込むために必要
import sys

# --- 3. ColabのシークレットからAPIキーを読み込む ---
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    if not api_key:
        raise ValueError("APIキーがシークレットに見つかりません。")
    genai.configure(api_key=api_key)
    print("✅ APIキーの設定が完了しました。")
except ValueError as e:
    print(f"エラー: {e}")
    print("Colabのシークレット（左側の鍵アイコン🔑）に 'GOOGLE_API_KEY' という名前でAPIキーを設定してください。")
    sys.exit(1) # スクリプトの実行を停止 (Colab環境ではセルが停止)
except Exception as e:
    print(f"予期せぬエラーが発生しました: {e}")
    sys.exit(1)

# --- 4. プロンプトとシステムプロンプトの定義 ---
# システムプロンプト: モデルの振る舞いや役割、応答形式などを指示します。
system_prompt = """
あなたは親切で知識豊富なAIアシスタント『コロ』です。
常に明るく、フレンドリーな口調で答えてください。
回答は日本語で行い、絵文字を適度に使うようにしてください。😄
技術的な質問には、ステップバイステップで、コード例も交えながら分かりやすく説明します。
応答の最後に、「コロがお手伝いしました✨ 何か他にあるかな？😊」と付け加えてください。
"""

# ユーザープロンプト: モデルに尋ねたい具体的な質問や指示です。
user_prompt = "Pythonでリスト内の数値を合計する簡単な方法を教えて！"

# --- 5. モデルの設定とAPI呼び出し ---
try:
    # 使用するモデルを選択 (例: 'gemini-1.5-flash-latest', 'gemini-1.5-pro-latest', 'gemini-pro')
    # 利用可能なモデルは変更される可能性があるため、ドキュメントを確認してください。
    model_name = 'gemini-1.5-flash-latest' # より高速なモデルに変更

    # GenerativeModelインスタンスを作成する際にsystem_instructionを指定
    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=system_prompt
    )

    print(f"\nモデル '{model_name}' に問い合わせています...")
    print("-" * 30)
    print(f"【システムプロンプト】\n{system_prompt}")
    print("-" * 30)
    print(f"【ユーザープロンプト】\n{user_prompt}")
    print("-" * 30)

    # コンテンツ生成を実行
    # generate_content にはユーザープロンプトのみを渡します
    response = model.generate_content(user_prompt)

    # --- 6. 結果の表示 ---
    print("【コロからの応答】")
    print("-" * 30)
    # response.text で生成されたテキストを取得
    print(response.text)

except genai.types.generation_types.BlockedPromptException as e:
    print("❌ エラー: プロンプトがブロックされました。")
    print("プロンプトの内容がセーフティポリシーに違反している可能性があります。内容を確認してください。")
    print(f"詳細: {e}")
except genai.types.generation_types.StopCandidateException as e:
    print("❌ エラー: 応答の生成が途中で停止しました。")
    print("コンテンツフィルターなどの理由で応答が完了しなかった可能性があります。")
    print(f"詳細: {e}")
except Exception as e:
    print(f"❌ エラーが発生しました: {e}")
    # APIキー関連やネットワーク、利用制限などのエラーメッセージを表示
    if "API key not valid" in str(e):
        print("APIキーが無効か、正しく設定されていない可能性があります。Colabシークレットの設定を確認してください。")
    elif "permission denied" in str(e) or "403" in str(e):
         print("APIキーに、選択したモデル ('{model_name}') を使用する権限がない可能性があります。Google Cloud ConsoleやAI Studioの設定を確認してください。")
    elif "Resource has been exhausted" in str(e) or "429" in str(e):
         print("APIの利用制限（例: 1分あたりのリクエスト数）に達した可能性があります。少し時間をおいてから再度試してください。")
    else:
        print("予期せぬAPIエラーが発生しました。")

In [ ]:
# Schema memo
'''
{
  "name": "Clinical_Review",
  "description": "Schema for clinical case summaries and associated queries",
  "strict": true,
  "schema": {
    "type": "object",
    "properties": {
      "usubjid": {
        "type": "string",
        "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)"
      },
      "timeline": {
        "type": ["array", "null"],
        "description": "Chronological events in the subject's case",
        "items": {
          "type": "object",
          "properties": {
            "date": {
              "type": "string",
              "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats."
            },
            "day": {
              "type": "integer",
              "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days."
            },
            "details": {
              "type": "string",
              "description": "Detailed information about the event (e.g., adverse event, medication administration)"
            }
          },
          "required": [
            "date",
            "day",
            "details"
          ],
          "additionalProperties": false
        }
      },
      "data_issues": {
        "type": ["array", "null"],
        "description": "List of data issues identified during review",
        "items": {
          "type": "object",
          "properties": {
            "issue_no": {
              "type": "integer",
              "description": "Unique issue number"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the issue",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "inconsistency": {
              "type": "string",
              "description": "Description of the data inconsistency"
            },
            "cause": {
              "type": "string",
              "description": "Suspected cause of the issue"
            },
            "resolution": {
              "type": "string",
              "description": "Proposed resolution for the issue"
            }
          },
          "required": [
            "issue_no",
            "variables",
            "inconsistency",
            "cause",
            "resolution"
          ],
          "additionalProperties": false
        }
      },
      "deviations": {
        "type": ["array", "null"],
        "description": "List of protocol deviations",
        "items": {
          "type": "object",
          "properties": {
            "deviation_no": {
              "type": "integer",
              "description": "Unique deviation number"
            },
            "impact": {
              "type": "string",
              "description": "Impact of the deviation on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the deviation",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            },
            "description": {
              "type": "string",
              "description": "Description of the deviation"
            },
            "protocol_reference": {
              "type": "string",
              "description": "Reference to the relevant section in the protocol"
            },
            "justification": {
              "type": "string",
              "description": "Justification for the deviation classification"
            }
          },
          "required": [
            "deviation_no",
            "impact",
            "variables",
            "description",
            "protocol_reference",
            "justification"
          ],
          "additionalProperties": false
        }
      },
      "queries": {
        "type": ["array", "null"],
        "description": "List of queries related to the subject",
        "items": {
          "type": "object",
          "properties": {
            "query_no": {
              "type": "integer",
              "description": "Unique query number"
            },
            "criticality": {
              "type": "string",
              "description": "Criticality of the query on the clinical trial results",
              "enum": ["Critical", "Major", "Minor"]
            },
            "inquiry": {
              "type": "string",
              "description": "Text of the inquiry to the study site"
            },
            "reason": {
              "type": "string",
              "description": "Justification for the inquiry"
            },
            "variables": {
              "type": "array",
              "description": "List of variables and their values related to the query",
              "items": {
                "type": "object",
                "properties": {
                  "variable": {
                    "type": "string",
                    "description": "Variable name"
                  },
                  "value": {
                    "type": "string",
                    "description": "Value of the variable"
                  }
                },
                "required": [
                  "variable",
                  "value"
                ],
                "additionalProperties": false
              }
            }
          },
          "required": [
            "query_no",
            "criticality",
            "inquiry",
            "reason",
            "variables"
          ],
          "additionalProperties": false
        }
      }
    },
    "required": [
      "usubjid",
      "timeline",
      "data_issues",
      "deviations",
      "queries"
    ],
    "additionalProperties": false
  }
}

''

# LLMへの送信 （Dify）

In [ ]:
!pip install sseclient-py
import requests
import sseclient
from IPython.display import display, Markdown

from google.colab import userdata
api_key = userdata.get('Dify_DatasetJSON')
user_id = 'JPMA_Sample'

## ワークフロー実行関数定義

In [ ]:
import json
import time
import requests
import sseclient

# 定数
DIFY_API_URL = 'https://api.dify.ai/v1/workflows/run'
CONTENT_TYPE_JSON = 'application/json'

def call_dify_api(api_key: str, payload: dict, stream: bool = False) -> requests.Response:
    """Dify APIを呼び出す共通関数"""
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': CONTENT_TYPE_JSON
    }
    try:
        response = requests.post(DIFY_API_URL, headers=headers, json=payload, stream=stream)
        response.raise_for_status()  # HTTPエラーが発生した場合に例外を発生させる
        return response
    except requests.exceptions.RequestException as e:
        print(f"API呼び出しエラー: {e}")
        raise

def run_dify_workflow(api_key: str, workflow_inputs: dict, user_id: str, streaming: bool = False) -> dict | sseclient.Event:
    """Difyワークフローを実行する

    Args:
        api_key: Dify APIキー
        workflow_inputs: ワークフローへの入力
        user_id: ユーザーID
        streaming: ストリーミングモードで実行するかどうか (Falseの場合はブロッキングモード)

    Returns:
        ストリーミングモードの場合はsseclient.Eventのイテレータ、
        ブロッキングモードの場合はAPIのレスポンスのJSONを辞書型で返す
    """
    payload = {
        'inputs': workflow_inputs,
        'response_mode': 'streaming' if streaming else 'blocking',
        'user': user_id
    }
    response = call_dify_api(api_key, payload, stream=streaming)
    if streaming:
        client = sseclient.SSEClient(response)
        return client.events()
    else:
        return response.json()

def safe_print_event_data(event: sseclient.Event):
    """
    与えられたSSEイベントデータから、存在する場合に特定の値を出力します。
    キーが存在しない場合は何も出力しません。Statusが存在する場合にのみTitleと結合させて表示します。

    Args:
        event: イベントデータを含むSSEイベントオブジェクト。event.data属性がJSON文字列であることを想定。
    """
    try:
        data = json.loads(event.data)

        if 'event' in data:
            print(f"Event: {data['event']}")

        if 'data' in data:
            title = data['data'].get('title')
            status = data['data'].get('status')
            if title is not None and status is not None:
                print(f"Node: {title} ({status})")
            elif title is not None:
                print(f"Node: {title}") # Statusが存在しない場合はTitleのみ表示

            if 'error' in data['data']:
                print(f"Error: {data['data']['error']}")
            if 'elapsed_time' in data['data']:
                print(f"Elapsed time: {data['data'].get('elapsed_time')}")
            if 'total_tokens' in data['data']:
                print(f"Total tokens: {data['data'].get('total_tokens')}")

    except json.JSONDecodeError as e:
        print(f"Error decoding JSON event data: {e}")
    except AttributeError as e:
        print(f"Error accessing event data attribute: {e}")

def run_workflow_with_retry(api_key: str, workflow_inputs: dict, user_id: str, max_retries: int = 3, retry_delay: int = 20):
    """ワークフローを実行し、エラー発生時にリトライを行う (ストリーミングモード専用)"""
    for retry in range(max_retries + 1):
        print(f"--- 試行回数: {retry + 1} ---")
        success = True
        try:
            for event in run_dify_workflow(api_key, workflow_inputs, user_id, streaming=True):
                safe_print_event_data(event)
                try:
                    event_data = json.loads(event.data)
                    if event_data.get('data', {}).get('error') is not None:
                        print(f"エラーが検出されました: {event_data['data']['error']}")
                        success = False
                        break
                except json.JSONDecodeError:
                    print("JSONデコードエラーが発生しました。")
                    success = False
                    break
                print('------')

            if success:
                print("ワークフローが正常に完了しました。")
                return json.loads(event.data)
            elif retry < max_retries:
                print(f"エラーが発生したため、{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

        except requests.exceptions.RequestException as e:
            print(f"APIリクエスト中にエラーが発生しました: {e}")
            success = False
            if retry < max_retries:
                print(f"{retry_delay}秒後に再試行します...")
                time.sleep(retry_delay)
            else:
                print("最大再試行回数に達しました。ワークフローは失敗しました。")
                return False

## ファイルアップロード関数定義

In [ ]:
import requests
import mimetypes

def upload_file_to_dify(api_key: str, file_path: str, user_id: str):
    """
    Difyにファイルをアップロードします。

    Args:
        api_key (str): Dify APIキー。
        file_path (str): アップロードするローカルファイルのパス。
        user_id (str): このファイルを関連付ける一意のエンドユーザー識別子。

    Returns:
        dict: APIからのレスポンス (JSON形式)。成功時はファイル情報が含まれます。
        None: エラーが発生した場合。
    """

    # APIエンドポイント
    upload_url = "https://api.dify.ai/v1/files/upload"

    mime_type, _ = mimetypes.guess_type(file_path)
    print(mime_type)

    try:
        # ファイルをバイナリモードで開く
        with open(file_path, 'rb') as f:
            # 'file'というキーでファイルオブジェクトを渡す
            files = {
                'file': (os.path.basename(file_path), f, mime_type) # (ファイル名, ファイルオブジェクト)
            }

            # POSTリクエストを送信
            response = requests.post(
                upload_url,
                headers = {"Authorization": f"Bearer {api_key}"},
                data={'user': user_id},
                files=files    # アップロードするファイル
            )
            print(files)

            # エラーレスポンスをチェック (4xx, 5xx)
            response.raise_for_status()

            # 成功した場合、JSONレスポンスを返す
            print(f"ファイル '{os.path.basename(file_path)}' のアップロードに成功しました。")
            return response.json()

    except FileNotFoundError:
        print(f"エラー: 指定されたファイルが見つかりません - {file_path}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"APIリクエスト中にエラーが発生しました: {e}")
        # エラーレスポンスの内容を表示しようと試みる
        try:
            print(f"サーバーからのエラー詳細: {response.text}")
        except NameError: # responseオブジェクトが存在しない場合
             pass
        except Exception as detail_e:
             print(f"サーバーからのエラー詳細の取得中に別のエラー: {detail_e}")
        return None
    except Exception as e:
        print(f"予期せぬエラーが発生しました: {e}")
        return None


### ファイルアップロードとワークフローのテスト

In [ ]:
#  # サンプルファイルの取得
#  !wget https://github.com/Takumi173/Test/releases/download/testdata/SampleText.txt
#
#  # アップロードしたいファイルのパス
#  file_to_upload = "SampleText.txt"
#
#  # アップロードの実行
#  upload_result = upload_file_to_dify(api_key, file_to_upload, user_id)
#
#  if upload_result:
#      print("\nアップロード結果:")
#      print(upload_result)
#
#      file_id = upload_result.get('id')
#      print(f"ファイルID: {file_id}")    # Workflowに投げるときはこのファイルIDを指定する
#  else:
#      print("\nアップロードに失敗しました。")
#
#
#  # アップロードしたファイルをワークフローに投げる
#  ModelName = 'gemini-2.0-flash'
#  workflow_inputs = {
#          'ModelName': ModelName,
#          'SysPrompt': '',
#          'UserInput': '以下にの内容を要約してください',
#          'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": file_id}
#    }
#
#  result = run_workflow_with_retry(api_key, workflow_inputs, user_id)
#  output_Task = result['data']['outputs']['text']
#  display(Markdown(output_Task))

## 出力フォーマットの定義

In [ ]:
# Markdown
Markdown_General = '''
**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。
'''
Taks1_Markdown = Markdown_General+'''
    1. 症例サマリー：[USUBJID]
        *   YYYY年MM月DD日 (Day XX): [有害事象、検査値、バイタルサインなどのイベントを、異常所見を中心に簡潔な文章で記載]

    2. 疑義事項: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks2_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせない疑義事項: [あり/なし]
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **疑義事項:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]
'''

Taks3_Markdown = Markdown_General+'''
    1. 確認した症例：[USUBJID]

    2. プロトコル逸脱: [あり/なし]
        *   **逸脱No.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **逸脱内容:** [具体的な逸脱内容を簡潔に記述。例：被験者XXXは、プロトコルで規定された投与量を超える量の治験薬を投与された]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

    3. 医療機関に問い合わせるクエリ: [あり/なし]
        *   **クエリNo.:**
            *   **臨床試験結果への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:**
            *   **判断理由:**
            *   **判断根拠:
                * [変数名 = 値; 判断するために参照した変数名と値をすべて記載する]

'''

# JSON
JSON_General = '''
**出力形式:**

*   すべての出力は指定されるJSON Schemaを用いて出力してください。
*   コードブロックや改行コードは使用せず、"{"で開始し、"}"で終わるJSONオブジェクト形式で出力してください。

**出力言語:**

*   JSONのValueは日本語で出力します

**JSON Schema**
'''

Task1_JSON = JSON_General+'''
{ "name": "Clinical_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "timeline": { "type": ["array", "null"], "description": "有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリー", "items": { "type": "object", "properties": { "date": { "type": "string", "description": "Date of the event.  Allows full (YYYY-MM-DD), partial (YYYY-MM), or year-only (YYYY) formats." }, "day": { "type": "integer", "description": "Day relative to study start (Day 1).  Allows negative values for pre-treatment days." }, "details": { "type": "string", "description": "異常所見を中心に簡潔な文章で記載する。正常範囲内の変動は省略可能。" } }, "required": [ "date", "day", "details" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "timeline", "queries" ], "additionalProperties": false } }
'''

Task2_JSON = JSON_General+'''
{ "name": "_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "data_issues": { "type": ["array", "null"], "description": "List of data issues identified during review", "items": { "type": "object", "properties": { "issue_no": { "type": "integer", "description": "Unique issue number" }, "variables": { "type": "array", "description": "List of variables and their values related to the issue", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "inconsistency": { "type": "string", "description": "具体的な矛盾の内容を記述" }, "cause": { "type": "string", "description": "問題点の原因（推測）" }, "resolution": { "type": "string", "description": "対応策（提案）" } }, "required": [ "issue_no", "variables", "inconsistency", "cause", "resolution" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "data_issues", "queries" ], "additionalProperties": false } }
'''

Task3_JSON = JSON_General+'''
{ "name": "Protocol_Deviation_Review", "description": "Schema for clinical case summaries and associated queries", "strict": true, "schema": { "type": "object", "properties": { "usubjid": { "type": "string", "description": "Unique subject identifier (e.g., STUDY-SITE-SUBJ)" }, "deviations": { "type": ["array", "null"], "description": "List of protocol deviations", "items": { "type": "object", "properties": { "deviation_no": { "type": "integer", "description": "Unique deviation number" }, "impact": { "type": "string", "description": "Impact of the deviation on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "variables": { "type": "array", "description": "List of variables and their values related to the deviation", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } }, "description": { "type": "string", "description": "具体的な逸脱内容を簡潔に記述。" }, "protocol_reference": { "type": "string", "description": "プロトコルの該当するセクション、ページ番号などを記載" }, "justification": { "type": "string", "description": "判断理由" } }, "required": [ "deviation_no", "impact", "variables", "description", "protocol_reference", "justification" ], "additionalProperties": false } }, "queries": { "type": ["array", "null"], "description": "List of queries related to the subject", "items": { "type": "object", "properties": { "query_no": { "type": "integer", "description": "Unique query number" }, "criticality": { "type": "string", "description": "Criticality of the query on the clinical trial results", "enum": ["Critical", "Major", "Minor"] }, "inquiry": { "type": "string", "description": "医療機関への問い合わせ文面" }, "reason": { "type": "string", "description": "判断理由" }, "variables": { "type": "array", "description": "List of variables and their values related to the query", "items": { "type": "object", "properties": { "variable": { "type": "string", "description": "Variable name" }, "value": { "type": "string", "description": "Value of the variable" } }, "required": [ "variable", "value" ], "additionalProperties": false } } }, "required": [ "query_no", "criticality", "inquiry", "reason", "variables" ], "additionalProperties": false } } }, "required": [ "usubjid", "deviations", "queries" ], "additionalProperties": false } }
'''


## プロンプトの作成

In [ ]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()


SysPrompt = '''
あなたは、臨床試験データのレビューを支援するAIアシスタントです。以下の前提知識を理解した上で、ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援してください。各タスクでは、ユーザープロンプトで指定された役割になりきって回答してください。

**前提知識:**

*   臨床試験においては患者の安全性が最優先され、有害事象の評価は特に重要です。
*   SDTM (Study Data Tabulation Model) は、CDISCによって策定された臨床試験データの標準モデルです。
*   Define.xmlはSDTMデータの構造を記述したメタデータファイルであり、参考情報として使用します。JSONデータ自体の内容、医学的妥当性、プロトコルとの整合性を優先してレビューしてください。
*   SDTMデータは、DM、AE、VS、LBなど、複数のドメイン（データセット）に分かれています。
*   報告されるJSONデータには、データ入力時の間違いが含まれる可能性があります。
*   提供された情報のみに基づいて回答を作成してください。想像やハルシネーションに基づいた回答は作成してはいけません。

**その他:**

*   指定された出力フォーマットに厳密に従って出力してください。
*   JSONデータまたはDefine.xmlの形式が不正な場合は、その旨をエラーメッセージとして出力してください。
'''




UserInput_Task1 = '''
あなたは臨床試験の専門医です。以下の指示に従い、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データのレビューとクエリ作成（必要な場合）を行ってください。

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、有害事象、検査値、バイタルサインなどの推移を時系列でまとめた症例サマリーを作成してください。
    *   特に、**異常所見**を中心に簡潔な文章で記載してください。正常範囲内の変動は省略して構いません。
    *   各イベントの日時は、Define.xmlに定義された日付変数などを参考に、正確に特定してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下のJSONデータのレビュー観点に基づき、JSONデータを改めて点検してください。
    *   医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *  **疑義事項がない場合は、クエリを作成する必要はありません。**「疑義事項なし」と回答してください。

*   **JSONデータのレビュー観点 (これらに限定されない):**
    *   **安全性:** 有害事象(AEドメイン)の報告内容は、医学的に妥当であるか？
    *   **医学的妥当性:** 検査値(LBドメイン)の変動、バイタルサイン(VSドメイン)の変動、併用薬(CMドメイン)との相互作用など、時間経過とともに医学的に問題となる点は見られるか？
    *   **有効性:** 特定された主要評価項目および副次評価項目について、その時間的変化は期待される効果と一致しているか？
    *   **その他:** 患者背景(DMドメイン)、既往歴(MHドメイン)、有害事象(AEドメイン)、治療歴(EXドメイン, CMドメイン)などを総合的に考慮し、時間経過を加味して安全性に懸念を生じる事項があれば記載してください。
    *   **プロトコル逸脱 (疑い):** 選択/除外基準、投与量、併用禁止薬、評価スケジュール、有害事象報告などについて、プロトコルからの逸脱の疑いがないか確認してください。（関連ドメイン: DM, MH, EX, CM, LB, VS, AEなど）
'''



UserInput_Task2 = '''
あなたはクリニカルデータマネージャーです。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、データ整合性レビューとクエリ作成（必要な場合）を行ってください。

**1. データ整合性レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、データの不整合が疑われる問題点を検出してください。
    *   **特に、以下の点に焦点を当ててレビューしてください。**
        *   **クロスドメイン整合性:** 異なるSDTMドメイン間で、データに矛盾がないか、ドメイン間の関連性が正しく表現されているか。
            *   **具体的な確認例 (これらに限定されない):**
                *   DM.SEXとAEにおける妊娠関連の有害事象
                *   AEの有害事象発現日や治験薬との関連性と、EXの治験薬の投与期間
                *   LBの検査値異常とAEの関連有害事象
                *   VSのバイタルサイン異常とAEの関連有害事象
                *   CM.CMTRTとAE/MHで報告されている疾患・既往歴との矛盾

        *   **単一ドメイン内の整合性:** Define.xmlの定義に照らして、矛盾なく解釈できるデータになっているか、プロトコルに照らしてデータの関連性が正しく表現されているか。
        *   **異常値:** Define.xmlで定義された範囲外、または医学的にありえない値がないか。
        *   **欠損値:** 欠損値の有無と理由（推測できる場合）。多い場合は原因を推測。
        *   **プロトコル逸脱 (データ品質の観点から):** データ入力/収集で、プロトコルからの逸脱（例：必須項目の未入力、不適切な時期のデータ収集）がないか。

    *   Define.xmlとデータの間に不整合がある場合は、「Define.xmlの修正候補」として報告してください。

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   データ整合性レビューの結果、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、臨床試験の評価項目に対する影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **疑義事項がない場合は、クエリを作成する必要はありません。**
'''



UserInput_Task3 = '''
あなたは、臨床試験の専門医、データマネージャー、CRAの視点を持つ、プロトコル遵守状況の確認者です。以下の指示に従い、提供される情報（JSONデータ、Define.xml、プロトコル）を基に、プロトコル逸脱の検出とクエリ作成（必要な場合）を行ってください。

**1. プロトコル逸脱の検出:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   JSONデータ、Define.xml、プロトコルを参照し、プロトコルからの逸脱を検出してください。
    *   Define.xmlは参考情報として活用し、データとプロトコルの内容を比較して逸脱を判断してください。
    *   **検出対象とすべき主要なプロトコル逸脱の例 (これらに限定されない):**
        *   **選択/除外基準違反:** (関連SDTMドメイン: DM, MH など)
        *   **投与量違反:** (関連SDTMドメイン: EX)
        *   **併用禁止薬の使用:** (関連SDTMドメイン: CM)
        *   **評価スケジュール違反:** (関連SDTMドメイン: LB, VS, その他)
        *   **有害事象報告違反**: (関連SDTMドメイン: AE)

**2. クエリの作成 (必要な場合のみ):**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   プロトコル逸脱を判定するために、医療機関への問い合わせが必要な事項（疑義、不明点、確認事項など）が発生した場合、その内容をまとめたクエリを作成してください。
    *   クエリは、報告されたデータと、Define.xml、プロトコルの記述に基づいて作成してください。提供された情報から逸脱する内容や、想像、ハルシネーションに基づくクエリは作成してはいけません。
    *   クエリは、プロトコル逸脱が臨床試験の評価項目に与える影響度を考慮し、重要度の高いものから優先的に作成してください。
    *   **プロトコル逸脱に関する疑義事項がない場合は、クエリを作成する必要はありません。**
'''


UserInput_end1 = '''\n---\n\n**データ:**\n\n*   臨床試験データ（JSON形式、SDTM準拠）:\n\n```json\n'''
UserInput_end2 = '''\n```\n\n*   データ定義ファイル（Define.xml）:\n\n```xml\n''' + define_xml + '''```\n'''

In [ ]:
with open('define_xml/define.xml', 'r') as f:
  define_xml = f.read()

SysPrompt_single = '''
あなたは、臨床試験データの正確性、完全性、医学的妥当性、プロトコル遵守状況のレビューを支援するAIアシスタントです。ユーザーからの指示（ユーザープロンプト）に従って、臨床試験データのレビューを支援します。

**最重要原則:**
*   **回答の主要な根拠は、提供された情報（JSONデータ、Define.xml、プロトコル）とします。** これらに基づき、客観的な事実を記述してください。
*   **医学的な妥当性の評価、潜在的なリスクの特定、データ間の関連性の解釈においては、あなたが持つ確立された一般的な医学知識を積極的に活用し、多角的な視点を提供してください。**
*   **一般的な医学知識に基づいて評価や指摘を行う場合は、その旨を明確に示してください** (例: 「一般的な医学的知見に基づくと、[薬剤A]と[薬剤B]の併用は[リスク]の可能性があるため注意が必要です。」)。
*   **いかなる場合も、提供されたデータや確立された一般的な医学知識に基づかない、個人的な意見、想像、推測、ハルシネーションに基づいた情報を生成してはいけません。** 患者の安全性を最優先し、有害事象の評価には特に注意を払ってください。

**前提知識:**
*   **SDTM (Study Data Tabulation Model):** CDISCによって策定された臨床試験データの標準モデルです。データはDM, AE, VS, LBなどのドメインに分かれており、レビューにはこれらの**ドメイン情報を横断的・統合的に評価する**必要があります。
*   **Define.xml:** SDTMデータの構造（変数名、ラベル、コードリスト、データ型など）を記述したメタデータファイルであり、データの意味を正確に理解するために**不可欠な情報源**です。JSONデータの解釈は、**必ずDefine.xmlの定義に基づいて**行ってください。
*   **データの不完全性:** 報告されるJSONデータには、データ入力時の間違いや不整合が含まれる可能性があることを理解しています。
*   **プロトコル:** 臨床試験の実施計画書であり、選択/除外基準、投与計画、評価スケジュール、有害事象報告手順などが規定されています。データのレビューはプロトコル遵守の観点からも行います。
*   **医学知識の活用:** あなたが持つ一般的な医学知識（例: 疾患、治療法、薬剤の作用・副作用・相互作用、生理学、検査値の臨床的意義に関する標準的な知識）は、データレビューにおいて重要な役割を果たします。**提供されたデータと照らし合わせながら、医学的な観点からの深い洞察や潜在的な懸念事項の指摘に活用してください。**

**タスク実行における注意:**
*   指定された**出力フォーマット**に厳密に従ってください。
*   提供された情報や一般的な医学知識をもってしてもタスクを実行できない場合（例：必要な情報が欠けている、矛盾が解決できない、専門性が高すぎる判断が必要な場合）、その旨を明確に指摘してください。

**エラーハンドリング:**
*   JSONデータ、Define.xml、またはプロトコルの形式が不正である、あるいは内容が著しく不足しておりレビューが困難な場合は、具体的な問題点を指摘し、処理を中断してください。例：「エラー：Define.xmlファイルが提供されていません。」、「エラー：JSONデータの[ドメイン名]に必要な変数[変数名]が含まれていません。」
'''

UserInput_single = '''
**役割:**

あなたは、**臨床試験データの多角的なレビュー担当者**です。**メディカルモニター、クリニカルデータマネージャー、およびプロトコル遵守確認者の視点を併せ持ち**、以下の指示に従って、提供される情報（プロトコル、JSONデータ、Define.xml）を基に、臨床試験データの統合レビュー、疑義事項の特定、およびクエリ/内部確認事項の作成（必要な場合）を行ってください。

**指示:**

**1. 症例サマリーの作成:**

*   **参照情報:** JSONデータ、Define.xml
*   **タスク:**
    *   JSONデータとDefine.xmlを参照し、患者の主要なイベントを時系列でまとめたサマリーを作成してください。
    *   **患者背景:** 最初にDMドメインから、主要な背景情報（例: 年齢、性別、人種など、Define.xmlで定義されたラベルを使用）を記載してください。
    *   **イベント推移:** 有害事象(AE)、検査値(LB)、バイタルサイン(VS)について、**異常変動**や**臨床的に注目すべき変化**を中心に記述してください。
        *   **異常・注目すべき変化の基準(例):**
            *   有害事象の発現、重症度・重篤度の変化、転帰
            *   検査値・バイタルサインの基準値からの逸脱 (Grade変化や明らかな異常値)
            *   ベースラインからの著しい変動
            *   正常範囲上限/下限付近での臨床的に意味のある変動
        *   **省略:** 原則として正常範囲内で臨床的に意義の小さい変動は省略してください。
    *   **日時:** 各イベントの日時は、関連する日付変数（例：AESTDY, LBDY, VSDYなど、Define.xml参照）に基づき特定し、**Study Day (--DY) を括弧内に併記**してください (例: `(Day 10)`)。
    *   **記述:** 簡潔な文章で客観的に記述してください。

**2. 統合レビュー:**

*   **参照情報:** JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   以下の**統合レビュー観点**に基づき、提供された情報を注意深く照合し、JSONデータを多角的にレビューしてください。問題点、矛盾、不整合、プロトコルからの逸脱の可能性などを検出・指摘してください。
    *   特定した各指摘事項およびそれに基づくクエリ/内部確認事項に対して、**後述の重要度の定義に基づき重要度（Critical/Major/Minor）を付与してください。** 判断は、臨床試験の評価項目や患者の安全性への潜在的な影響度、データの信頼性への影響度を総合的に考慮してください。

*   **統合レビュー観点:**
    *   **【医学的妥当性・安全性】 (メディカルモニター視点)**
        *   **AE:** 有害事象の報告内容（事象名、重篤度、重症度、治験薬との関連性、処置、転帰）に、他のデータ（LB, VS, CMなど）との矛盾や医学的な観点から不自然な点はないか？特に重篤な有害事象の評価は、関連データと照らして一貫性があるか？
        *   **LB/VS:** 検査値やバイタルサインの変動パターン（異常値、経時変化）に、医学的に懸念される点はないか？ 他の臨床情報（AE, CM, MHなど）と整合しているか？
        *   **CM/AE/MH:** 併用薬と有害事象/既往歴との関連、潜在的な薬物相互作用に関して、プロトコルや一般的な医学知識に基づき、注意すべき点はないか？
        *   **総合評価:** 患者背景(DM)、既往歴(MH)、有害事象(AE)、治療歴(EX, CM)などを総合的に考慮し、時間経過を踏まえて、患者の安全性に関する潜在的な懸念事項はないか？

    *   **【データ整合性】 (データマネージャー視点)**
        *   **クロスドメイン整合性:** 異なるドメイン間のデータに矛盾はないか？ (例: AE発生日 vs LB/VS測定日、AE回復日 vs LB/VS測定日、AE vs CM開始/終了日、MH vs AE/CM、DM.SEX vs 性別依存のイベント/検査、AE発現日 vs EX投与期間)
        *   **ドメイン内整合性:** 各ドメイン内のデータに矛盾はないか？ (例: AE 開始日 <= AE 終了日、投与量と単位の一貫性)
        *   **異常値/外れ値:** 医学的/現実的にありえない値、Define.xmlで定義された範囲外の値はないか？
        *   **欠損値:** 重要な変数に欠損はないか？ (例: AE関連性、LB/VS結果、主要評価項目) 欠損が許容されるか、理由が適切か？

    *   **【プロトコル遵守】 (プロトコル確認者視点)**
        *   **選択/除外基準:** 患者はプロトコルで規定された選択基準を満たし、除外基準に該当していないか？ (DM, MHなどを参照)
        *   **治験薬投与:** 投与量、投与経路、投与期間などはプロトコルで規定された通りか？ (EXを参照)
        *   **併用禁止/制限薬:** プロトコルで禁止または制限されている薬剤が使用されていないか？ (CMを参照)
        *   **評価スケジュール/手順:** 検査や評価はプロトコルで規定されたタイミングと手順で実施されているか？ (VISIT情報、各ドメインの--DY/VISITNUMなどを参照)
        *   **有害事象報告:** 有害事象（特に重篤な有害事象）はプロトコルの規定に従って報告されているか？ (AEを参照)

**3. 疑義事項の分類とクエリ/内部確認事項の作成:**

*   **参照情報:** 統合レビューの結果、JSONデータ、Define.xml、プロトコル
*   **タスク:**
    *   **統合レビューで特定された問題点や疑義事項についてのみ**、以下の分類を行ってください。
        *   **医療機関へのクエリ:** 医療機関への問い合わせが必要な事項。
        *   **内部確認事項:** 医療機関への問い合わせは不要だが、内部で確認・記録すべき事項（例：軽微なデータ不整合、解釈の確認）。

    *   特定した各指摘事項およびそれに基づくクエリ/内部確認事項に対して、**後述の重要度の定義に基づき重要度（Critical/Major/Minor）を付与してください。** 判断は、臨床試験の評価項目や患者の安全性への潜在的な影響度、データの信頼性への影響度を総合的に考慮してください。
    *   レビューの結果、クエリや内部確認事項を作成する必要がない場合は、「疑義事項なし」と明確に回答してください。


**重要度の定義:**
*   **Critical (致命的/重大):**
    *   **影響:** 患者の権利、安全性、健康に**重大なリスク**をもたらす、またはその可能性がある。データの**信頼性や完全性を著しく損ない**、試験結果（特に主要評価項目）の解釈に**重大な影響**を与える。規制当局への報告義務やGCP遵守に**重大な影響**を与える。
    *   **具体的な状況例:**
        *   重篤な有害事象 (SAE) の未報告、報告遅延、または評価（重篤性、関連性等）に関する重大な疑義。
        *   主要な選択/除外基準の明確な違反。
        *   治験薬の重大な誤投与（過量、併用禁忌薬との併用等）。
        *   主要評価項目データの欠損、重大な不整合、または信頼性への疑義。
        *   同意取得前の治験関連手順の実施。
    *   **対応:** 通常、即時のアクション（例: 緊急クエリ、プロトコル逸脱報告）が必要。

*   **Major (主要):**
    *   **影響:** 患者の権利、安全性、健康に**潜在的なリスク**をもたらす可能性がある（Criticalほどではない）。データの**信頼性や完全性に影響**を与え、試験結果（特に副次評価項目）の解釈に**影響を与える可能性**がある。プロトコルからの**重要な逸脱**に該当する。
    *   **具体的な状況例:**
        *   非重篤な有害事象の評価（重症度、関連性等）と他の臨床データとの明らかな矛盾。
        *   副次評価項目や重要な安全性評価項目に関連するデータの不整合や欠損。
        *   併用禁止/制限薬の使用（安全性リスクが中程度以下）。
        *   重要な検査・評価の未実施や、規定されたVisit Windowからの逸脱。
        *   投与量変更や一時中断/再開に関する記録の不備や矛盾。
    *   **対応:** 通常、クエリ発行による確認・修正や、内部での詳細な調査が必要。

*   **Minor (軽微):**
    *   **影響:** 患者の安全性や試験結果の解釈への**直接的な影響は小さい**と考えられる。主にデータの**品質や一貫性**に関わる問題。プロトコルからの**軽微な逸脱**で、試験の主要な目的に大きな影響を与えないもの。
    *   **具体的な状況例:**
        *   明らかな誤字脱字（ただし、事象名や薬剤名など、解釈に影響を与えうる場合はMajor以上と判断することもある）。
        *   重要度の低いデータの欠損や軽微な不整合（例: 終了日が開始日より前だが、臨床的な時間経過から明らかに誤記と判断でき、影響が小さい）。
        *   臨床的に意義の小さい検査値/バイタルサインの記録に関する軽微な矛盾。
        *   Visit日付のわずかなずれ（プロトコルで許容範囲が定義されていない場合など）。
    *   **対応:** 内部確認事項として記録するか、他のクエリと併せて確認する、または修正不要と判断する場合もある。


**出力形式:** 以下のテンプレートに従ってMarkdown形式で出力してください。

# [USUBJID]のデータ統合レビュー報告

## 1. 症例サマリー

*   **患者背景:**

[DMドメインから取得した主要な背景情報を簡潔に記載]

*   **イベント推移:**
    *   [YYYY年MM月DD日 (Day XX): イベント内容 (例: 有害事象「頭痛」(Severe) 発現)]
    *   [YYYY年MM月DD日 (Day YY): イベント内容 (例: ALT値上昇 (Grade 1, 基準値上限の1.5倍))]
    *   ... (時系列で記載) ...

## 2. 統合レビュー結果

*   **医学的観点からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** M-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **内容:** [具体的な医学的懸念事項や妥当性に関する指摘]
            *   **根拠:** [判断の根拠となった変数名 = 値、プロトコルの記述などを記載]
        *   ... (複数の指摘事項があれば M-2, M-3...)

*   **データ整合性観点からの指摘事項:**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** D-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **内容:** [具体的なデータの不整合、異常値、欠損値などに関する指摘]
            *   **根拠:** [判断の根拠となった変数名 = 値、Define.xmlの記述などを記載]
            *   **(Define.xml修正候補):** [必要であれば記載]
        *   ... (複数の指摘事項があれば D-2, D-3...)

*   **プロトコル遵守観点からの指摘事項 (逸脱の可能性):**
    *   [指摘事項がない場合は「指摘事項なし」と記載]
    *   (指摘事項がある場合)
        *   **指摘No.:** P-1
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **逸脱の可能性:** [具体的なプロトコルからの逸脱の可能性]
            *   **プロトコル該当箇所:** [プロトコルの該当するセクション、ページ番号などを記載]
            *   **根拠:** [判断の根拠となった変数名 = 値などを記載]
        *   ... (複数の指摘事項があれば P-2, P-3...)

## 3. 疑義事項

*   [クエリも内部確認事項もない場合は「疑義事項なし」と記載]
*   **医療機関へのクエリ:**
    *   [クエリがない場合は「クエリなし」と記載]
    *   (クエリがある場合)
        *   **クエリNo.:** Q-1 (関連指摘No.: [例: M-1, D-2])
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **医療機関への問い合わせ文面:** [具体的かつ客観的な問い合わせ内容。例: "Day 10の有害事象「XXXX」の重症度(AE.AESEV)がGrade 2と報告されていますが、同日の臨床検査(LB)では基準値の範囲内です。XXXXの重症度および転機についてご確認ください。"]
            *   **判断理由:** [なぜ問い合わせが必要かの簡潔な理由。例: 有害事象の評価と関連する処置の整合性を確認するため。]
            *   **判断根拠:**
                *   [変数名 = 値; 例: AE.AETERM = '頭痛', AE.AESTDY = 10, AE.AESEV = 'MODERATE (Grade 2)']
                *   [変数名 = 値; 例: CM.CMTRT where CMSTDY = 10 (該当レコードなし)]
                *   [プロトコル該当箇所: 必要であれば記載]
        *   ... (複数のクエリがあれば Q-2, Q-3...)

*   **内部確認事項 (問い合わせ不要):**
    *   [内部確認事項がない場合は「内部確認事項なし」と記載]
    *   (内部確認事項がある場合)
        *   **確認事項No.:** I-1 (関連指摘No.: [例: D-1])
            *   **臨床試験結果/安全性への影響度合い:** [Critical/Major/Minor]
            *   **疑義事項/確認内容:** [問い合わせは不要だが、記録・確認すべき内容。例: DMドメインの生年月日(BRTHDTC)が一部不明瞭('19XX')だが、年齢(AGE)は計算されており、選択基準を満たしているため現時点では問い合わせ不要と判断。記録として残す。]
            *   **判断理由:** [なぜ問い合わせ不要か、なぜ記録が必要かの簡潔な理由。例: 年齢情報は他の変数で補完されており、選択基準の確認は可能。データの完全性の観点から記録。]
            *   **判断根拠:**
                *   [変数名 = 値; 例: DM.USUBJID = 'XXX', DM.BRTHDTC = '19XX', DM.AGE = 55]
                *   [プロトコル該当箇所: 例: Section 4.1 選択基準 (年齢 18-75歳)]
                *   [Define.xml該当箇所: 必要であれば記載]
            *   ... (複数の内部確認事項があれば I-2, I-3...)

'''

UserInput_single_end1 = '''\n---\n\n**臨床試験データ（JSON形式、SDTM準拠）:**\n\n```json\n'''
UserInput_single_end2 = '''\n```\n\n**データ定義ファイル（Define.xml）:**\n\n```xml\n''' + define_xml + '''```\n'''

In [ ]:
def create_workflow_input(ModelName, SysPrompt, UserInput_Task, UserInput_end1, datasetjson, UserInput_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_Task + UserInput_end1 + datasetjson + UserInput_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

In [ ]:
def create_workflow_input_single(ModelName, SysPrompt_single, UserInput_single, UserInput_single_end1, datasetjson, UserInput_single_end2):
    return {
        'ModelName': ModelName,
        'SysPrompt': SysPrompt,
        'UserInput': UserInput_single + UserInput_single_end1 + datasetjson + UserInput_single_end2,
        'AttachedFile': {"type": "document", "transfer_method": "local_file", "upload_file_id": "6b06d4f8-d47a-441f-bf67-d8700f76f556"}
    }

## 実行

In [ ]:
# ModelNameの設定
#ModelName = 'gemini-2.0-flash'
#ModelName = 'gemini-2.0-flash-exp'
#ModelName = 'gemini-2.0-flash-exp-multi'
ModelName = 'gemini-2.0-pro-exp'
#ModelName = 'gemini-2.0-pro-exp-02-05'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21'
#ModelName = 'gemini-2.0-flash-thinking-exp-01-21-multi'
#ModelName = 'gemini-2.0-flash-thinking-exp'
#ModelName = 'gemini-2.0-flash-thinking-exp-multi'


# データ更新症例の抽出
updated_subjects = []
for l in Target_data:
  updated_subjects.append(l[1])

updated_subjects = sorted(list(set(updated_subjects)))
print(updated_subjects)
print(len(updated_subjects))


In [ ]:
# output mode: Markdown / JSON
Output_Format = "Markdown"

if Output_Format == "Markdown":
  UserInput_Task1 = UserInput_Task1 + Taks1_Markdown
  UserInput_Task2 = UserInput_Task2 + Taks2_Markdown
  UserInput_Task3 = UserInput_Task3 + Taks3_Markdown
elif Output_Format == "JSON":
  UserInput_Task1 = UserInput_Task1 + Task1_JSON
  UserInput_Task2 = UserInput_Task2 + Task2_JSON
  UserInput_Task3 = UserInput_Task3 + Task3_JSON


In [ ]:
import pandas as pd

results_list = []

updated_subjects = ['01-704-1017','01-703-1042','01-701-1111',]

for subj in updated_subjects:
    datasetjson = filter_data(dataset_list_updated, subj)
    print(f"処理完了：'datasetjson' に USUBJID が {subj} のデータを出力しました。")

    row_data = {'Subject': subj}  # 各行のデータを格納する辞書

#    # Task 1 の処理
#    workflow_inputs_Task1 = create_workflow_input(ModelName, SysPrompt, UserInput_Task1, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task1 = run_workflow_with_retry(api_key, workflow_inputs_Task1, user_id)
#        output_Task1 = result_Task1['data']['outputs']['text']
#        display(Markdown(output_Task1))
#        row_data['Task1'] = output_Task1
#    except Exception as e:
#        print(f"Task 1 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task1'] = "Error"
#
#    # Task 2 の処理
#    workflow_inputs_Task2 = create_workflow_input(ModelName, SysPrompt, UserInput_Task2, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task2 = run_workflow_with_retry(api_key, workflow_inputs_Task2, user_id)
#        output_Task2 = result_Task2['data']['outputs']['text']
#        display(Markdown(output_Task2))
#        row_data['Task2'] = output_Task2
#    except Exception as e:
#        print(f"Task 2 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task2'] = "Error"
#
#    # Task 3 の処理
#    workflow_inputs_Task3 = create_workflow_input(ModelName, SysPrompt, UserInput_Task3, UserInput_end1, json.dumps(datasetjson), UserInput_end2)
#    try:
#        result_Task3 = run_workflow_with_retry(api_key, workflow_inputs_Task3, user_id)
#        output_Task3 = result_Task3['data']['outputs']['text']
#        display(Markdown(output_Task3))
#        row_data['Task3'] = output_Task3
#    except Exception as e:
#        print(f"Task 3 でエラーが発生しました (Subject: {subj}): {e}")
#        row_data['Task3'] = "Error"
#
    # 統合プロンプトの処理
    workflow_inputs_single = create_workflow_input_single(ModelName, SysPrompt_single, UserInput_single, UserInput_single_end1, json.dumps(datasetjson), UserInput_single_end2)
    try:
        result_Task_single = run_workflow_with_retry(api_key, workflow_inputs_single, user_id)
        output_Task_single = result_Task_single['data']['outputs']['text']
        display(Markdown(output_Task_single))
        row_data['Task_single'] = output_Task_single
    except Exception as e:
        print(f"統合プロンプトの処理でエラーが発生しました (Subject: {subj}): {e}")
        row_data['Task_single'] = "Error"
    results_list.append(row_data)

# DataFrameを作成
df_results = pd.DataFrame(results_list)

# DataFrameを表示
display(df_results)

## 結果の出力

### JSONの場合の出力関数

In [ ]:
import re
import json

def extract_json(text):
    # ```json ... ``` のパターンを検索 (非貪欲マッチ)
    match = re.search(r'```json\s*([\s\S]*?)\s*```', text)

    if match:
        json_string = match.group(1)
        try:
            data = json.loads(json_string)
            return data
        except json.JSONDecodeError:
            print("Error: Invalid JSON found.")
            return None
    else:
        print("Error: No JSON code block found.")
        return None

# Test
#extract_json(df_results['Task1'][0])


In [ ]:
import json
from datetime import datetime

# --- 新しいヘルパー関数 ---
def format_partial_date(date_str):
    """
    部分日付を含む日付文字列を可能な限り指定の日本語形式にフォーマットする。
    対応形式: YYYY-MM-DD, YYYY-MM, YYYY
    """
    if not date_str:
        return '日付不明'

    # 優先度順にフォーマットを試す
    formats_map = {
        '%Y-%m-%d': '%Y年%m月%d日',
        '%Y-%m': '%Y年%m月',
        '%Y': '%Y年'
    }

    for input_format, output_format in formats_map.items():
        try:
            date_obj = datetime.strptime(date_str, input_format)
            return date_obj.strftime(output_format)
        except ValueError:
            continue # 次のフォーマットを試す

    # どの形式にも一致しない場合は、元の文字列をそのまま返す
    # (予期しない形式や "UNKNOWN" などの文字列に対応するため)
    return date_str
# --- ヘルパー関数ここまで ---

def format_variables_list(variables):
    """
    変数リストをMarkdownの箇条書き形式の複数行文字列にフォーマットする関数
    各行は '* 変数名 = 値' の形式
    """
    if not variables:
        return [] # 空のリストを返す
    lines = [f"* {v.get('variable', 'N/A')} = {v.get('value', 'N/A')}" for v in variables]
    return lines

def indent_lines(lines, indent_spaces):
    """指定された行リストの各行にインデントを追加する"""
    indent = " " * indent_spaces
    return "\n".join([indent + line for line in lines])

def generate_queries_section(queries, section_title="医療機関に問い合わせるクエリ", no_label="クエリNo."):
    """
    クエリセクションのMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    markdown = []
    has_queries = bool(queries) # None や空リストでないかチェック

    markdown.append(f"{section_title}: {'あり' if has_queries else 'なし'}")
    if has_queries:
        for query in queries:
            markdown.append(f"    *   **{no_label}:** {query.get('query_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {query.get('criticality', 'N/A')}")
            markdown.append(f"        *   **医療機関への問い合わせ文面:** {query.get('inquiry', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {query.get('reason', 'N/A')}")
            markdown.append(f"        *   **判断根拠:**")
            variable_lines = format_variables_list(query.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

# --- generate_timeline_markdown を修正 ---
def generate_timeline_markdown(data):
    """
    'timeline' キーが存在する場合のMarkdownを生成する関数 (部分日付対応)
    """
    usubjid = data.get('usubjid', 'N/A')
    timeline = data.get('timeline', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 症例サマリー：{usubjid}")

    for entry in timeline:
        # 新しいヘルパー関数を使って日付をフォーマット
        formatted_date = format_partial_date(entry.get('date'))

        day = entry.get('day', '不明') # Day は日付形式に関わらず表示
        details = entry.get('details', '詳細不明')
        markdown.append(f"    *   {formatted_date} (Day {day}): {details}")

    markdown.append("") # 空行
    markdown.append(generate_queries_section(queries, section_title="2. 疑義事項", no_label="クエリNo."))

    return "\n".join(markdown)
# --- generate_timeline_markdown の修正ここまで ---

def generate_data_issues_markdown(data):
    """
    'data_issues' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    data_issues = data.get('data_issues', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="2. 医療機関に問い合わせるクエリ", no_label="クエリNo."))
    markdown.append("") # 空行

    has_data_issues = bool(data_issues)
    markdown.append(f"3. 医療機関に問い合わせない疑義事項: {'あり' if has_data_issues else 'なし'}")
    if has_data_issues:
        for issue in data_issues:
            markdown.append(f"    *   **疑義No.:** {issue.get('issue_no', 'N/A')}")
            markdown.append(f"        *   **疑義事項:** {issue.get('inconsistency', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {issue.get('cause', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(issue.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))

    return "\n".join(markdown)

def generate_deviations_markdown(data):
    """
    'deviations' キーが存在する場合のMarkdownを生成する関数 (判断根拠を箇条書き表示)
    """
    usubjid = data.get('usubjid', 'N/A')
    deviations = data.get('deviations', [])
    queries = data.get('queries')

    markdown = []
    markdown.append(f"1. 確認した症例：{usubjid}")
    markdown.append("") # 空行

    has_deviations = bool(deviations)
    markdown.append(f"2. プロトコル逸脱: {'あり' if has_deviations else 'なし'}")
    if has_deviations:
        for deviation in deviations:
            markdown.append(f"    *   **逸脱No.:** {deviation.get('deviation_no', 'N/A')}")
            markdown.append(f"        *   **臨床試験結果への影響度合い:** {deviation.get('impact', 'N/A')}")
            markdown.append(f"        *   **逸脱内容:** {deviation.get('description', 'N/A')}")
            markdown.append(f"        *   **プロトコル該当箇所:** {deviation.get('protocol_reference', 'N/A')}")
            markdown.append(f"        *   **判断理由:** {deviation.get('justification', 'N/A')}")
            markdown.append(f"        *   **判断根拠:")
            variable_lines = format_variables_list(deviation.get('variables', []))
            if variable_lines:
                markdown.append(indent_lines(variable_lines, 12))
    markdown.append("") # 空行

    markdown.append(generate_queries_section(queries, section_title="3. 医療機関に問い合わせるクエリ", no_label="クエリNo."))

    return "\n".join(markdown)

def json_to_markdown(json_input):
    """
    JSONデータを受け取り、指定の形式のMarkdownに変換するメイン関数 (部分日付対応)
    """
    try:
        if isinstance(json_input, str):
            data = json.loads(json_input)
        elif isinstance(json_input, dict):
            data = json_input
        else:
            return "エラー: 入力はJSON文字列またはPython辞書である必要があります。"
    except json.JSONDecodeError:
        return "エラー: 無効なJSON文字列です。"
    except Exception as e:
        return f"エラー: 予期せぬエラーが発生しました - {e}"

    if 'timeline' in data:
        return generate_timeline_markdown(data)
    elif 'data_issues' in data:
        return generate_data_issues_markdown(data)
    elif 'deviations' in data:
        return generate_deviations_markdown(data)
    else:
        usubjid = data.get('usubjid', 'N/A')
        queries = data.get('queries')
        markdown = []
        markdown.append(f"1. 確認した症例：{usubjid}")
        markdown.append("\n---\n")
        markdown.append("入力データには timeline, data_issues, deviations のいずれのキーも含まれていません。")
        if queries is not None:
             markdown.append("\n---\n")
             markdown.append(generate_queries_section(queries, section_title="クエリ情報", no_label="クエリNo."))
        return "\n".join(markdown)

# Test
#markdown_output = json_to_markdown(extract_json(df_results['Task3'][0]))
#print(markdown_output)


In [ ]:
def result_output_JSON(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = json_to_markdown(extract_json(row['Task1']))
        task2 = json_to_markdown(extract_json(row['Task2']))
        task3 = json_to_markdown(extract_json(row['Task3']))

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_JSON(df_results)
#print(output_text)

### Markdownの場合の出力関数

In [ ]:

def result_output_Markdown(df: pd.DataFrame) -> str:
    """
    DataFrameを指定されたテキスト形式に変換します。

    Args:
        df: 変換するDataFrame。カラム名は 'Subject', 'Task1', 'Task2', 'Task3' である必要があります。

    Returns:
        変換後のテキストデータ。
    """
    text_data = ""
    for index, row in df.iterrows():
        subject = row['Subject']
        task1 = row['Task1']
        task2 = row['Task2']
        task3 = row['Task3']

        text_data += f"# {subject}\n"
        text_data += f"## Task1: Clinical Review Results\n"
        text_data += f"{task1}\n"
        text_data += f"## Task2: DM Review Results\n"
        text_data += f"{task2}\n"
        text_data += f"## Task3: Protocol Deviation Review Results\n"
        text_data += f"{task3}\n\n"

    return text_data

#output_text = result_output_Markdown(df_results)

### 出力

In [ ]:
if Output_Format == "Markdown":
  output_text = result_output_Markdown(df_results)
elif Output_Format == "JSON":
  output_text = result_output_JSON(df_results)

# mdファイルに保存
output_file = 'output_' + ModelName + '.md'  # 保存するファイル名を指定
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(output_text)

print(output_text)